In [ ]:
from src.dune_swarm import DuneSwarm

# Modelo homogéneo (lambda2_std=0 → réplica exacta de Robson & Baas)

print("unimodal")
model = DuneSwarm(n_dunes_init=20, qsat=100, lambda2_std=0.0, seed=42, wind_regime='unimodal')
for _ in range(5):
    model.step()
    print(model.wind_vec)

df = model.datacollector.get_model_vars_dataframe()

print('bimodal_acute')
# Modelo heterogéneo (extensión original)
model2 = DuneSwarm(n_dunes_init=20, qsat=100, lambda2_std=0.5, seed=42, wind_regime='bimodal_acute')
for _ in range(5):
    model2.step()
    print(model2.wind_vec)

df2 = model2.datacollector.get_model_vars_dataframe()

In [ ]:
df2

In [ ]:
df

In [ ]:
model2.steps

In [ ]:
dunes2 = model2.datacollector.get_agent_vars_dataframe()
single_dune = dunes2.xs(key = 1, level = 'AgentID')
single_dune.plot(y='rw')

In [ ]:
import pandas as pd

# Load the entire Parquet file
df = pd.read_parquet('../resultados/summary.parquet')

In [ ]:
df.qsat.plot.hist()

In [ ]:
len(df)

In [ ]:
import sqlite3
import pandas as pd

con = sqlite3.connect("../resultados/dunas.db")

df = pd.read_sql("""
    SELECT 
        r.q0ratio,
        r.qshift_ratio,
        COUNT(a.rowid)          AS agent_rows,
        COUNT(DISTINCT a.step)  AS steps_saved,
        COUNT(DISTINCT a.agent_id) AS unique_agents
    FROM agent_data a
    JOIN runs r ON a.run_id = r.run_id
    GROUP BY r.q0ratio, r.qshift_ratio
    ORDER BY agent_rows DESC
""", con)

con.close()
df

In [ ]:
import shapely
print(shapely.__version__)

In [ ]:
import subprocess
import sys
result = subprocess.run(
    [sys.executable, "-c", "import shapely; print(shapely.__version__)"],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

In [ ]:
import sys
sys.path.insert(0, ".")
from src.dune_swarm import DuneSwarm

m = DuneSwarm(
    **{
        "qsat": 30.0, "q0ratio": 0.1, "qshift_ratio": 0.0,
        "lambda2_std": 0.0, "wind_regime": "unimodal",
        "dt": 0.1, "n_steps_init": 0,
        "simwidth": 5000.0, "simlength": 5000.0,
        "fieldwidth": 4000.0, "fieldlength": 1000.0,
        "inject": True, "inject_mode": "weq", "rho0": 4e-3,
        "n_dunes_init": 0, "collisions": True,
        "outflux_mode": "Duran", "seed": 42,
        # resto de FIXED_PARAMS
        "lambda1": 1.0, "lambda2_mean": 1.8, "lambda3": 1/3,
        "alpha": 0.05, "delta": 4.6, "c": 45.0, "w0": 2.0,
    }
)

for step in [1, 10, 50, 100]:
    while m.current_step < step:
        m.step()
    agents = list(m.agents)
    ys = [a.pos[1] for a in agents]
    print(f"Paso {step:>3} | N={len(agents):>4} | "
          f"y_min={min(ys):.1f} y_max={max(ys):.1f} y_mean={sum(ys)/len(ys):.1f}")

In [ ]:

import sys
from pathlib import Path
sys.path.insert(0, str(Path(".").resolve()))
from src.dune_swarm import DuneSwarm

casos = [
    {"nombre": "LENTO  qsat=30  q0=0.10", "qsat": 30.0,  "q0ratio": 0.10, "n_steps": 2089},
    {"nombre": "RAPIDO qsat=120 q0=0.50", "qsat": 120.0, "q0ratio": 0.50, "n_steps": 114},
    {"nombre": "SININY qsat=30  q0=0.00", "qsat": 30.0,  "q0ratio": 0.00, "n_steps": 300},
]

for caso in casos:
    m = DuneSwarm(
        qsat=caso["qsat"], q0ratio=caso["q0ratio"], qshift_ratio=0.0,
        lambda2_std=0.0, wind_regime="unimodal",
        dt=0.5,
        simwidth=2000.0, simlength=3000.0,
        fieldwidth=1000.0, fieldlength=500.0,
        inject=True, inject_mode="weq", rho0=2e-4,
        n_dunes_init=0, collisions=True,
        outflux_mode="Duran", seed=42,
        lambda1=1.0, lambda2_mean=1.8, lambda3=1/3,
        alpha=0.05, delta=4.6, c=45.0, w0=2.0,
    )

    n_steps = caso["n_steps"]
    checkpoints = [1, n_steps//4, n_steps//2, n_steps]

    print(f"\n{'='*60}")
    print(f"  {caso['nombre']}  |  n_steps={n_steps}")
    print(f"{'='*60}")

    for target in checkpoints:
        while m.current_step < target:
            m.step()
        agents = list(m.agents)
        if agents:
            ys = [a.pos[1] for a in agents]
            ws = [a.width for a in agents]
            print(f"  Paso {target:>5} | N={len(agents):>4} | "
                  f"y_min={min(ys):>7.1f}  y_mean={sum(ys)/len(ys):>7.1f} | "
                  f"W_mean={sum(ws)/len(ws):>6.1f}m")
        else:
            print(f"  Paso {target:>5} | Sin agentes")

m = DuneSwarm(
    qsat=120.0, q0ratio=0.50, qshift_ratio=0.0,
    lambda2_std=0.0, wind_regime="unimodal",
    dt=0.5, simwidth=2000.0, simlength=3000.0,
    fieldwidth=1000.0, fieldlength=500.0,
    inject=True, inject_mode="weq", rho0=2e-4,
    n_dunes_init=0, collisions=True,
    outflux_mode="Duran", seed=42,
    lambda1=1.0, lambda2_mean=1.8, lambda3=1/3,
    alpha=0.05, delta=4.6, c=45.0, w0=2.0,
)

m.step()
agents = list(m.agents)
for a in agents[:3]:
    print(f"pos={a.pos}  lw={a.lw:.2f}  rw={a.rw:.2f}  "
          f"migration_vec={a._migration_vec}  width={a.width:.2f}")
    

    m2 = DuneSwarm(
    qsat=120.0, q0ratio=0.50, qshift_ratio=0.0,
    lambda2_std=0.0, wind_regime="unimodal",
    dt=0.5, simwidth=2000.0, simlength=3000.0,
    fieldwidth=1000.0, fieldlength=500.0,
    inject=True, inject_mode="weq", rho0=2e-4,
    n_dunes_init=0, collisions=True,
    outflux_mode="Duran", seed=42,
    lambda1=1.0, lambda2_mean=1.8, lambda3=1/3,
    alpha=0.05, delta=4.6, c=45.0, w0=2.0,
)

# Interceptar distribute_flux para ver el vector antes de step()
from src import flux_physics as fp
_orig = fp.distribute_flux

def _patched(*args, **kwargs):
    _orig(*args, **kwargs)
    agents = args[0]
    for a in agents[:3]:
        print(f"  [post-distribute] id={a.unique_id} "
              f"migration_vec={a._migration_vec}  pos={a.pos}")

fp.distribute_flux = _patched
import src.dune_swarm
src.dune_swarm.distribute_flux = _patched

m2.step()
fp.distribute_flux = _orig

m2 = DuneSwarm(
    qsat=120.0, q0ratio=0.50, qshift_ratio=0.0,
    lambda2_std=0.0, wind_regime="unimodal",
    dt=0.5, simwidth=2000.0, simlength=3000.0,
    fieldwidth=1000.0, fieldlength=500.0,
    inject=True, inject_mode="weq", rho0=2e-4,
    n_dunes_init=0, collisions=True,
    outflux_mode="Duran", seed=42,
    lambda1=1.0, lambda2_mean=1.8, lambda3=1/3,
    alpha=0.05, delta=4.6, c=45.0, w0=2.0,
)

# Antes de step — ver estado inicial
agents = list(m2.agents)
print("ANTES de step:")
for a in agents[:3]:
    print(f"  id={a.unique_id} pos={a.pos} migration_vec={a._migration_vec}")

# Llamar distribute_flux manualmente
from src.flux_physics import distribute_flux
distribute_flux(
    agents=agents,
    wind_vec=m2._wind_vec,
    qsat=m2.qsat,
    q0=m2.q0,
    dt=m2.dt,
    w0=m2.w0,
    c=m2.c,
    alpha=m2.alpha,
    delta=m2.delta,
    outflux_mode=m2.outflux_mode,
)

print("\nDESPUES de distribute_flux:")
for a in agents[:3]:
    print(f"  id={a.unique_id} pos={a.pos} migration_vec={a._migration_vec}")

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path(".").resolve()))

from src.dune_swarm import DuneSwarm
from src.flux_physics import distribute_flux

m2 = DuneSwarm(
    qsat=120.0, q0ratio=0.50, qshift_ratio=0.0,
    lambda2_std=0.0, wind_regime="unimodal",
    dt=0.5, simwidth=2000.0, simlength=3000.0,
    fieldwidth=1000.0, fieldlength=500.0,
    inject=True, inject_mode="weq", rho0=2e-4,
    n_dunes_init=0, collisions=True,
    outflux_mode="Duran", seed=42,
    lambda1=1.0, lambda2_mean=1.8, lambda3=1/3,
    alpha=0.05, delta=4.6, c=45.0, w0=2.0,
)

# Primero corremos un paso para que se inyecten dunas
m2.step()
print(f"Agentes después de step 1: {len(list(m2.agents))}")

# Ahora reseteamos manualmente el wind_vec y probamos distribute_flux
m2._wind_vec = (0.0, -1.0)
agents = list(m2.agents)

# Resetear migration_vec de todos
for a in agents:
    a._migration_vec = (0.0, 0.0)

print("\nANTES de distribute_flux:")
for a in agents[:3]:
    print(f"  id={a.unique_id} pos={a.pos}  "
          f"lw={a.lw:.2f}  rw={a.rw:.2f}  "
          f"migration_vec={a._migration_vec}")

# Llamar distribute_flux manualmente
distribute_flux(
    agents=agents,
    wind_vec=m2._wind_vec,
    qsat=m2.qsat,
    q0=m2.q0,
    dt=m2.dt,
    w0=m2.w0,
    c=m2.c,
    alpha=m2.alpha,
    delta=m2.delta,
    outflux_mode=m2.outflux_mode,
)

print("\nDESPUES de distribute_flux:")
for a in agents[:3]:
    print(f"  id={a.unique_id} pos={a.pos}  "
          f"lw={a.lw:.2f}  rw={a.rw:.2f}  "
          f"migration_vec={a._migration_vec}")

In [ ]:
import numpy as np
import pandas as pd

# Parámetros físicos fijos
c      = 45.0
w0     = 2.0
delta  = 4.6
alpha  = 0.05
CFL    = 0.1
factor = 3

# Dominio propuesto
simlength  = 3000.0
fieldwidth = 1000.0

# Grid completo
qsats         = [30.0, 60.0, 90.0, 120.0]
q0ratios      = [0.0, 0.10, 0.20, 0.30, 0.50]
qshift_ratios = [0.0, 0.10, 0.20, 0.30, 0.50]
lambda2_stds  = [0.0, 0.25, 0.5, 1.0]

# Para la tabla solo variamos qsat y q0ratio (los que determinan W_eq y v_mig)
# qshift y lambda2_std no afectan dt ni n_steps
rows = []
for qsat in qsats:
    for q0ratio in q0ratios:

        denom = q0ratio * qsat - alpha * qsat
        if denom <= 0:
            w_eq         = None
            inject_active = False
            # Sin inyección → dunas iniciales pequeñas
            # Usamos w_min como referencia para CFL
            w_min = (delta / 2) / (1 - alpha)
            w_ref = w_min * 3   # tamaño inicial típico
            v_mig = c * qsat / (w_ref + w0)
        else:
            w_eq          = delta * qsat / denom
            v_mig         = c * qsat / (w_eq + w0)
            w_ref         = w_eq
            inject_active = True

        # dt adaptativo por CFL
        dt      = CFL * w_ref / v_mig
        # Simplificación: dt = CFL * (w_ref + w0) / (c * qsat)
        dt_check = CFL * (w_ref + w0) / (c * qsat)

        # Verificación: d_paso = v_mig * dt debe ser CFL * w_ref
        d_paso  = v_mig * dt

        # n_steps para cruzar factor veces el dominio
        t_cruce = simlength / v_mig
        t_total = t_cruce * factor
        n_steps = int(np.ceil(t_total / dt))

        # Inyección por paso
        n_inject = (2e-4 * v_mig * fieldwidth) if inject_active else 0.0

        # Tiempo estimado de cómputo (referencia: ~0.01s por paso × n_agentes)
        n_agentes_est = min(n_inject * 100, 150) if inject_active else 10
        t_computo_s   = n_steps * n_agentes_est * 0.01

        rows.append({
            "qsat":          qsat,
            "q0ratio":       q0ratio,
            "w_eq":          round(w_eq, 1) if w_eq else "-",
            "v_mig":         round(v_mig, 2),
            "dt":            round(dt, 4),
            "d_paso_m":      round(d_paso, 3),
            "d_paso/w_ref":  round(d_paso / w_ref, 3),
            "t_total_años":  round(t_total, 1),
            "n_steps":       n_steps,
            "n_inject/paso": round(n_inject, 2),
            "t_computo_s":   round(t_computo_s, 1),
            "inject":        inject_active,
        })

df = pd.DataFrame(rows)

print("=" * 110)
print(f"CFL={CFL} | factor={factor}× | simlength={simlength:.0f}m | fieldwidth={fieldwidth:.0f}m")
print("=" * 110)
print(df.to_string(index=False))

print("\n\nResumen:")
print(f"  dt       mín : {df['dt'].min():.4f} años  "
      f"(qsat={df.loc[df['dt'].idxmin(),'qsat']:.0f}, "
      f"q0ratio={df.loc[df['dt'].idxmin(),'q0ratio']})")
print(f"  dt       máx : {df['dt'].max():.4f} años  "
      f"(qsat={df.loc[df['dt'].idxmax(),'qsat']:.0f}, "
      f"q0ratio={df.loc[df['dt'].idxmax(),'q0ratio']})")
print(f"  n_steps  mín : {df['n_steps'].min():>6,}  "
      f"(qsat={df.loc[df['n_steps'].idxmin(),'qsat']:.0f}, "
      f"q0ratio={df.loc[df['n_steps'].idxmin(),'q0ratio']})")
print(f"  n_steps  máx : {df['n_steps'].max():>6,}  "
      f"(qsat={df.loc[df['n_steps'].idxmax(),'qsat']:.0f}, "
      f"q0ratio={df.loc[df['n_steps'].idxmax(),'q0ratio']})")
print(f"  n_steps medio: {df['n_steps'].mean():>6.0f}")
print(f"\n  Corridas sin inyección : {len(df[~df['inject']])} de {len(df)}")
print(f"  t_computo estimado total (400 corridas): "
      f"{df['t_computo_s'].sum() * len(qshift_ratios) * len(lambda2_stds) / 3600:.1f} horas")

In [ ]:
import numpy as np
import pandas as pd

# Parámetros físicos fijos
c      = 45.0
w0     = 2.0
delta  = 4.6
alpha  = 0.05
CFL    = 0.1

# Dominio propuesto
simlength  = 3000.0
fieldwidth = 1000.0

# Grid completo
qsats         = [30.0, 60.0, 90.0, 120.0]
q0ratios      = [0.0, 0.10, 0.20, 0.30, 0.50]
qshift_ratios = [0.0, 0.10, 0.20, 0.30, 0.50]
lambda2_stds  = [0.0, 0.25, 0.5, 1.0]

N_STEPS_MAX = 2000

rows = []
for qsat in qsats:
    for q0ratio in q0ratios:

        denom = q0ratio * qsat - alpha * qsat
        if denom <= 0:
            w_eq          = None
            inject_active = False
            factor        = 1       # sin inyección converge en 1 cruce
            w_min         = (delta / 2) / (1 - alpha)
            w_ref         = w_min * 3
            v_mig         = c * qsat / (w_ref + w0)
        else:
            w_eq          = delta * qsat / denom
            v_mig         = c * qsat / (w_eq + w0)
            w_ref         = w_eq
            inject_active = True
            factor        = 3

        # dt adaptativo CFL
        dt     = CFL * w_ref / v_mig
        d_paso = v_mig * dt

        # n_steps con cap
        t_cruce      = simlength / v_mig
        t_total      = t_cruce * factor
        n_steps_raw  = int(np.ceil(t_total / dt))
        n_steps      = min(n_steps_raw, N_STEPS_MAX)
        capped       = n_steps < n_steps_raw

        # Inyección por paso
        n_inject = (2e-4 * v_mig * fieldwidth) if inject_active else 0.0

        # Tiempo estimado cómputo
        n_agentes_est = min(n_inject * 100, 150) if inject_active else 10
        t_computo_s   = n_steps * n_agentes_est * 0.01

        rows.append({
            "qsat":          qsat,
            "q0ratio":       q0ratio,
            "w_eq":          round(w_eq, 1) if w_eq else "-",
            "v_mig":         round(v_mig, 2),
            "dt":            round(dt, 4),
            "d_paso_m":      round(d_paso, 3),
            "factor":        factor,
            "n_steps_raw":   n_steps_raw,
            "n_steps":       n_steps,
            "capped":        "✓" if capped else "",
            "n_inject/paso": round(n_inject, 2),
            "t_computo_s":   round(t_computo_s, 1),
            "inject":        inject_active,
        })

df = pd.DataFrame(rows)

print("=" * 120)
print(f"CFL={CFL} | n_steps_max={N_STEPS_MAX} | simlength={simlength:.0f}m | fieldwidth={fieldwidth:.0f}m")
print("=" * 120)
print(df.to_string(index=False))

print("\n\nResumen:")
print(f"  dt       mín : {df['dt'].min():.4f} años  "
      f"(qsat={df.loc[df['dt'].idxmin(),'qsat']:.0f}, "
      f"q0ratio={df.loc[df['dt'].idxmin(),'q0ratio']})")
print(f"  dt       máx : {df['dt'].max():.4f} años  "
      f"(qsat={df.loc[df['dt'].idxmax(),'qsat']:.0f}, "
      f"q0ratio={df.loc[df['dt'].idxmax(),'q0ratio']})")
print(f"  n_steps  mín : {df['n_steps'].min():>6,}  "
      f"(qsat={df.loc[df['n_steps'].idxmin(),'qsat']:.0f}, "
      f"q0ratio={df.loc[df['n_steps'].idxmin(),'q0ratio']})")
print(f"  n_steps  máx : {df['n_steps'].max():>6,}  "
      f"(qsat={df.loc[df['n_steps'].idxmax(),'qsat']:.0f}, "
      f"q0ratio={df.loc[df['n_steps'].idxmax(),'q0ratio']})")
print(f"  n_steps medio: {df['n_steps'].mean():>6.0f}")
print(f"  Corridas capeadas: {len(df[df['capped']=='✓'])} de {len(df)}")
print(f"  Corridas sin inyección: {len(df[~df['inject']])} de {len(df)}")
print(f"\n  t_computo estimado total (400 corridas): "
      f"{df['t_computo_s'].sum() * len(qshift_ratios) * len(lambda2_stds) / 3600:.1f} horas")

In [ ]:
import numpy as np
import pandas as pd

# Parámetros físicos fijos
c      = 45.0
w0     = 2.0
delta  = 4.6
alpha  = 0.05
CFL    = 0.1

# Dominio propuesto
simlength  = 3000.0
fieldwidth = 500.0
rho0       = 1.6e-3

# Grid completo
qsats         = [30.0, 60.0, 90.0, 120.0]
q0ratios      = [0.0, 0.10, 0.20, 0.30, 0.50]
qshift_ratios = [0.0, 0.10, 0.20, 0.30, 0.50]
lambda2_stds  = [0.0, 0.25, 0.5, 1.0]

N_STEPS_MAX = 2000

rows = []
for qsat in qsats:
    for q0ratio in q0ratios:

        denom = q0ratio * qsat - alpha * qsat
        if denom <= 0:
            w_eq          = None
            inject_active = False
            factor        = 1
            w_min         = (delta / 2) / (1 - alpha)
            w_ref         = w_min * 3
            v_mig         = c * qsat / (w_ref + w0)
        else:
            w_eq          = delta * qsat / denom
            v_mig         = c * qsat / (w_eq + w0)
            w_ref         = w_eq
            inject_active = True
            factor        = 3

        # dt adaptativo CFL
        dt     = CFL * w_ref / v_mig
        d_paso = v_mig * dt

        # n_steps con cap
        t_cruce     = simlength / v_mig
        t_total     = t_cruce * factor
        n_steps_raw = int(np.ceil(t_total / dt))
        n_steps     = min(n_steps_raw, N_STEPS_MAX)
        capped      = n_steps < n_steps_raw

        # Inyección por paso
        n_inject = (rho0 * v_mig * fieldwidth) if inject_active else 0.0

        # Tiempo estimado cómputo
        n_agentes_est = min(n_inject * 100, 150) if inject_active else 10
        t_computo_s   = n_steps * n_agentes_est * 0.01

        rows.append({
            "qsat":          qsat,
            "q0ratio":       q0ratio,
            "w_eq":          round(w_eq, 1) if w_eq else "-",
            "v_mig":         round(v_mig, 2),
            "dt":            round(dt, 4),
            "d_paso_m":      round(d_paso, 3),
            "factor":        factor,
            "n_steps_raw":   n_steps_raw,
            "n_steps":       n_steps,
            "capped":        "✓" if capped else "",
            "n_inject/paso": round(n_inject, 2),
            "t_computo_s":   round(t_computo_s, 1),
            "inject":        inject_active,
        })

df = pd.DataFrame(rows)

print("=" * 120)
print(f"CFL={CFL} | n_steps_max={N_STEPS_MAX} | rho0={rho0:.1e} | "
      f"simlength={simlength:.0f}m | fieldwidth={fieldwidth:.0f}m")
print("=" * 120)
print(df.to_string(index=False))

print("\n\nResumen:")
print(f"  dt       mín : {df['dt'].min():.4f} años  "
      f"(qsat={df.loc[df['dt'].idxmin(),'qsat']:.0f}, "
      f"q0ratio={df.loc[df['dt'].idxmin(),'q0ratio']})")
print(f"  dt       máx : {df['dt'].max():.4f} años  "
      f"(qsat={df.loc[df['dt'].idxmax(),'qsat']:.0f}, "
      f"q0ratio={df.loc[df['dt'].idxmax(),'q0ratio']})")
print(f"  n_steps  mín : {df['n_steps'].min():>6,}  "
      f"(qsat={df.loc[df['n_steps'].idxmin(),'qsat']:.0f}, "
      f"q0ratio={df.loc[df['n_steps'].idxmin(),'q0ratio']})")
print(f"  n_steps  máx : {df['n_steps'].max():>6,}  "
      f"(qsat={df.loc[df['n_steps'].idxmax(),'qsat']:.0f}, "
      f"q0ratio={df.loc[df['n_steps'].idxmax(),'q0ratio']})")
print(f"  n_steps medio: {df['n_steps'].mean():>6.0f}")
print(f"  Corridas capeadas    : {len(df[df['capped']=='✓'])} de {len(df)}")
print(f"  Corridas sin inyecc. : {len(df[~df['inject']])} de {len(df)}")
print(f"\n  t_computo estimado total (400 corridas): "
      f"{df['t_computo_s'].sum() * len(qshift_ratios) * len(lambda2_stds) / 3600:.1f} horas")

In [ ]:
import numpy as np
import pandas as pd

# Parámetros físicos fijos
c      = 45.0
w0     = 2.0
delta  = 4.6
alpha  = 0.05
CFL    = 0.1

# Dominio propuesto
simlength  = 3000.0
fieldwidth = 500.0
rho0       = 1.6e-3

# Grid completo
qsats         = [30.0, 60.0, 90.0, 120.0]
q0ratios      = [0.0, 0.10, 0.20, 0.30, 0.50]
qshift_ratios = [0.0, 0.10, 0.20, 0.30, 0.50]
lambda2_stds  = [0.0, 0.25, 0.5, 1.0]

N_STEPS_MAX = 2000

rows = []
for qsat in qsats:
    for q0ratio in q0ratios:

        denom = q0ratio * qsat - alpha * qsat
        if denom <= 0:
            w_eq          = None
            inject_active = False
            factor        = 1
            w_min         = (delta / 2) / (1 - alpha)
            w_ref         = w_min * 3
            v_mig         = c * qsat / (w_ref + w0)
        else:
            w_eq          = delta * qsat / denom
            v_mig         = c * qsat / (w_eq + w0)
            w_ref         = w_eq
            inject_active = True
            factor        = 3

        # dt adaptativo CFL
        dt     = CFL * w_ref / v_mig
        d_paso = v_mig * dt

        # n_steps con cap
        t_cruce     = simlength / v_mig
        t_total     = t_cruce * factor
        n_steps_raw = int(np.ceil(t_total / dt))
        n_steps     = min(n_steps_raw, N_STEPS_MAX)
        capped      = n_steps < n_steps_raw

        # Inyección por paso
        n_inject = (rho0 * v_mig * fieldwidth) if inject_active else 0.0

        # Tiempo estimado cómputo
        n_agentes_est = min(n_inject * 100, 150) if inject_active else 10
        t_computo_s   = n_steps * n_agentes_est * 0.01

        rows.append({
            "qsat":          qsat,
            "q0ratio":       q0ratio,
            "w_eq":          round(w_eq, 1) if w_eq else "-",
            "v_mig":         round(v_mig, 2),
            "dt":            round(dt, 4),
            "d_paso_m":      round(d_paso, 3),
            "factor":        factor,
            "n_steps_raw":   n_steps_raw,
            "n_steps":       n_steps,
            "capped":        "✓" if capped else "",
            "n_inject/paso": round(n_inject, 2),
            "t_computo_s":   round(t_computo_s, 1),
            "inject":        inject_active,
        })

df = pd.DataFrame(rows)

print("=" * 120)
print(f"CFL={CFL} | n_steps_max={N_STEPS_MAX} | rho0={rho0:.1e} | "
      f"simlength={simlength:.0f}m | fieldwidth={fieldwidth:.0f}m")
print("=" * 120)
print(df.to_string(index=False))

print("\n\nResumen:")
print(f"  dt       mín : {df['dt'].min():.4f} años  "
      f"(qsat={df.loc[df['dt'].idxmin(),'qsat']:.0f}, "
      f"q0ratio={df.loc[df['dt'].idxmin(),'q0ratio']})")
print(f"  dt       máx : {df['dt'].max():.4f} años  "
      f"(qsat={df.loc[df['dt'].idxmax(),'qsat']:.0f}, "
      f"q0ratio={df.loc[df['dt'].idxmax(),'q0ratio']})")
print(f"  n_steps  mín : {df['n_steps'].min():>6,}  "
      f"(qsat={df.loc[df['n_steps'].idxmin(),'qsat']:.0f}, "
      f"q0ratio={df.loc[df['n_steps'].idxmin(),'q0ratio']})")
print(f"  n_steps  máx : {df['n_steps'].max():>6,}  "
      f"(qsat={df.loc[df['n_steps'].idxmax(),'qsat']:.0f}, "
      f"q0ratio={df.loc[df['n_steps'].idxmax(),'q0ratio']})")
print(f"  n_steps medio: {df['n_steps'].mean():>6.0f}")
print(f"  Corridas capeadas    : {len(df[df['capped']=='✓'])} de {len(df)}")
print(f"  Corridas sin inyecc. : {len(df[~df['inject']])} de {len(df)}")
print(f"\n  t_computo estimado total (400 corridas): "
      f"{df['t_computo_s'].sum() * len(qshift_ratios) * len(lambda2_stds) / 3600:.1f} horas")

In [ ]:
import sqlite3
con = sqlite3.connect("../resultados/dunas.db")

# Ver params de una corrida fallida de cada tipo
for run_id in ["run_75463af9", "run_ffa303dd"]:
    row = con.execute(
        "SELECT params_json FROM runs WHERE run_id=?", (run_id,)
    ).fetchone()
    if row:
        import json
        p = json.loads(row[0])
        print(f"\n{run_id}:")
        print(f"  qsat={p.get('qsat')} q0ratio={p.get('q0ratio')} "
              f"qshift={p.get('qshift_ratio')} dt={p.get('dt')}")

con.close()

In [ ]:
import glob
logs = sorted(glob.glob("../resultados/run_grid_*.txt"))
print(logs[-1])  # el mas reciente

In [ ]:
import sys, traceback
from pathlib import Path
sys.path.insert(0, str(Path(".").resolve()))

from src.dune_swarm import DuneSwarm
from scripts.run_storage import RunStorage

# Caso 1: invalid literal — qsat=30, q0ratio=0.0
params1 = {
    "qsat": 30.0, "q0ratio": 0.0, "qshift_ratio": 0.0,
    "lambda2_std": 0.0, "wind_regime": "unimodal",
    "dt": 0.004983, "n_dunes_init": 0, "collisions": True,
    "outflux_mode": "Duran", "seed": 980330,
    "lambda1": 1.0, "lambda2_mean": 1.8, "lambda3": 1/3,
    "alpha": 0.05, "delta": 4.6, "c": 45.0, "w0": 2.0,
    "simwidth": 2000.0, "simlength": 3000.0,
    "fieldwidth": 500.0, "fieldlength": 500.0,
    "inject": True, "inject_mode": "weq", "rho0": 4e-3,
}

# Caso 2: division by zero — qsat=30, q0ratio=0.1
params2 = {**params1, "q0ratio": 0.1, "dt": 0.6406}

for nombre, params in [("CASO1 q0=0.0", params1), ("CASO2 q0=0.1", params2)]:
    print(f"\n{'='*50}")
    print(f"  {nombre}")
    print(f"{'='*50}")
    try:
        storage = RunStorage("resultados/")
        run_id  = storage.new_run(params)
        model   = DuneSwarm(**params)
        for _ in range(3):
            model.step()
        storage.finalize_run(run_id, model, update_summary=False)
        print("OK - sin errores")
    except Exception:
        traceback.print_exc()

In [ ]:
import sys, traceback
from pathlib import Path
sys.path.insert(0, str(Path(".").resolve()))


params2 = {
    "qsat": 30.0, "q0ratio": 0.1, "qshift_ratio": 0.1,
    "lambda2_std": 0.0, "wind_regime": "unimodal",
    "dt": 0.6406, "n_dunes_init": 0, "collisions": True,
    "outflux_mode": "Duran", "seed": 980330,
    "lambda1": 1.0, "lambda2_mean": 1.8, "lambda3": 1/3,
    "alpha": 0.05, "delta": 4.6, "c": 45.0, "w0": 2.0,
    "simwidth": 2000.0, "simlength": 3000.0,
    "fieldwidth": 500.0, "fieldlength": 500.0,
    "inject": True, "inject_mode": "weq", "rho0": 4e-3,
}

try:
    storage = RunStorage("resultados/")
    run_id  = storage.new_run(params2)
    model   = DuneSwarm(**params2)
    for _ in range(3):
        model.step()
    storage.finalize_run(run_id, model, update_summary=False)
    print("OK")
except Exception:
    traceback.print_exc()

In [ ]:
import sys, traceback
from pathlib import Path
sys.path.insert(0, str(Path(".").resolve()))
sys.path.insert(0, str(Path("scripts").resolve()))

# Reimportar para asegurar que carga la version nueva
import importlib
import scripts.run_storage
importlib.reload(scripts.run_storage)

from src.dune_swarm import DuneSwarm
from scripts.run_storage import RunStorage

# Verificar que el fix esta presente
import inspect
src = inspect.getsource(RunStorage._save_agent_data)
print("Fix presente:" , "Normalizar a MultiIndex" in src)

# Reproducir caso fallido
params = {
    "qsat": 30.0, "q0ratio": 0.0, "qshift_ratio": 0.0,
    "lambda2_std": 0.0, "wind_regime": "unimodal",
    "dt": 0.004983, "n_dunes_init": 0, "collisions": True,
    "outflux_mode": "Duran", "seed": 980330,
    "lambda1": 1.0, "lambda2_mean": 1.8, "lambda3": 1/3,
    "alpha": 0.05, "delta": 4.6, "c": 45.0, "w0": 2.0,
    "simwidth": 2000.0, "simlength": 3000.0,
    "fieldwidth": 500.0, "fieldlength": 500.0,
    "inject": True, "inject_mode": "weq", "rho0": 4e-3,
}

try:
    storage = RunStorage("resultados_test/")
    run_id  = storage.new_run(params)
    model   = DuneSwarm(**params)
    for _ in range(5):
        model.step()
    storage.finalize_run(run_id, model, update_summary=False)
    print("OK - sin errores")
except Exception:
    traceback.print_exc()

In [ ]:
import sys, traceback
from pathlib import Path
sys.path.insert(0, str(Path(".").resolve()))
sys.path.insert(0, str(Path("scripts").resolve()))


# Caso division by zero — q0ratio=0.1, qshift=0.1
params = {
    "qsat": 30.0, "q0ratio": 0.1, "qshift_ratio": 0.1,
    "lambda2_std": 0.0, "wind_regime": "unimodal",
    "dt": 0.6406, "n_dunes_init": 0, "collisions": True,
    "outflux_mode": "Duran", "seed": 980330,
    "lambda1": 1.0, "lambda2_mean": 1.8, "lambda3": 1/3,
    "alpha": 0.05, "delta": 4.6, "c": 45.0, "w0": 2.0,
    "simwidth": 2000.0, "simlength": 3000.0,
    "fieldwidth": 500.0, "fieldlength": 500.0,
    "inject": True, "inject_mode": "weq", "rho0": 4e-3,
}

try:
    storage = RunStorage("resultados_test/")
    run_id  = storage.new_run(params)
    model   = DuneSwarm(**params)
    for i in range(979):  # n_steps completo
        model.step()
        if i % 100 == 0:
            print(f"Paso {i} | N={len(list(model.agents))}", flush=True)
    storage.finalize_run(run_id, model, update_summary=False)
    print("OK")
except Exception:
    traceback.print_exc()

In [ ]:
import sqlite3, json

con = sqlite3.connect("resultados/dunas.db")

# Corridas fallidas con division by zero — tomar las primeras 3
rows = con.execute("""
    SELECT run_id, params_json FROM runs 
    WHERE n_steps_run IS NULL
    LIMIT 20
""").fetchall()
con.close()

for run_id, params_json in rows:
    p = json.loads(params_json)
    print(f"{run_id}: qsat={p.get('qsat')} q0ratio={p.get('q0ratio')} "
          f"qshift={p.get('qshift_ratio')} lambda2_std={p.get('lambda2_std')} "
          f"dt={p.get('dt'):.4f}")

In [ ]:
params = {
    "qsat": 30.0, "q0ratio": 0.0, "qshift_ratio": 0.0,
    "lambda2_std": 0.0, "wind_regime": "unimodal",
    "dt": 0.005, "n_dunes_init": 0, "collisions": True,
    "outflux_mode": "Duran", "seed": 980330,
    "lambda1": 1.0, "lambda2_mean": 1.8, "lambda3": 1/3,
    "alpha": 0.05, "delta": 4.6, "c": 45.0, "w0": 2.0,
    "simwidth": 2000.0, "simlength": 3000.0,
    "fieldwidth": 500.0, "fieldlength": 500.0,
    "inject": True, "inject_mode": "weq", "rho0": 4e-3,
}

try:
    storage = RunStorage("resultados_test/")
    run_id  = storage.new_run(params)
    model   = DuneSwarm(**params)
    for i in range(2000):
        model.step()
    storage.finalize_run(run_id, model, update_summary=False)
    print("OK")
except Exception:
    traceback.print_exc()

In [ ]:
params = {
    "qsat": 30.0, "q0ratio": 0.0, "qshift_ratio": 0.0,
    "lambda2_std": 0.0, "wind_regime": "unimodal",
    "dt": 0.005, "n_dunes_init": 0, "collisions": True,
    "outflux_mode": "Duran", "seed": 980330,
    "lambda1": 1.0, "lambda2_mean": 1.8, "lambda3": 1/3,
    "alpha": 0.05, "delta": 4.6, "c": 45.0, "w0": 2.0,
    "simwidth": 2000.0, "simlength": 3000.0,
    "fieldwidth": 500.0, "fieldlength": 500.0,
    "inject": True, "inject_mode": "weq", "rho0": 4e-3,
}

# Probar todas las combinaciones de qshift y lambda2_std con q0ratio=0
import itertools
for qshift, l2std in itertools.product(
    [0.0, 0.10, 0.20, 0.30, 0.50],
    [0.0, 0.25, 0.5, 1.0]
):
    p = {**params, "qshift_ratio": qshift, "lambda2_std": l2std}
    try:
        model = DuneSwarm(**p)
        for i in range(2000):
            model.step()
        print(f"OK  qshift={qshift} l2std={l2std}")
    except Exception as e:
        print(f"ERR qshift={qshift} l2std={l2std} paso={model.current_step} → {e}")

In [ ]:
import itertools

for qsat, qshift, l2std in itertools.product(
    [30.0, 60.0, 90.0, 120.0],
    [0.0, 0.10, 0.20, 0.30, 0.50],
    [0.0, 0.25, 0.5, 1.0]
):
    p = {**params, 
         "qsat": qsat, "q0ratio": 0.1,
         "qshift_ratio": qshift, "lambda2_std": l2std,
         "dt": 0.6406 if qsat == 30 else 0.3203 if qsat == 60 else 0.2135 if qsat == 90 else 0.1601,
    }
    try:
        model = DuneSwarm(**p)
        for i in range(979):
            model.step()
        print(f"OK  qsat={qsat} qshift={qshift} l2std={l2std}")
    except Exception as e:
        print(f"ERR qsat={qsat} qshift={qshift} l2std={l2std} "
              f"paso={model.current_step} → {e}")

In [ ]:
import traceback

p = {**params,
     "qsat": 30.0, "q0ratio": 0.1,
     "qshift_ratio": 0.2, "lambda2_std": 0.0,
     "dt": 0.6406,
}

try:
    model = DuneSwarm(**p)
    for i in range(979):
        model.step()
except Exception:
    print(f"Falló en paso {model.current_step}")
    traceback.print_exc()

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path(".").resolve()))
sys.path.insert(0, str(Path("scripts").resolve()))

from src.dune_swarm import DuneSwarm
import matplotlib.pyplot as plt

params_base = {
    "qsat": 30.0, "q0ratio": 0.1,
    "qshift_ratio": 0.0, "lambda2_std": 0.0,
    "dt": 0.6406, "n_dunes_init": 0, "collisions": True,
    "outflux_mode": "Duran", "seed": 980330,
    "lambda1": 1.0, "lambda2_mean": 1.8, "lambda3": 1/3,
    "alpha": 0.05, "delta": 4.6, "c": 45.0, "w0": 2.0,
    "simwidth": 2000.0, "simlength": 3000.0,
    "fieldwidth": 500.0, "fieldlength": 500.0,
    "inject": True, "inject_mode": "weq", "rho0": 4e-3,
}

model = DuneSwarm(**params_base)
for _ in range(979):
    model.step()

df = model.datacollector.get_model_vars_dataframe()

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
df["N_dunes"].plot(ax=axes[0], title="N dunas")
df["mean_width"].plot(ax=axes[1], title="Ancho medio (m)")
df["collision_count"].plot(ax=axes[2], title="Colisiones acumuladas")
plt.tight_layout()
plt.show()

print(f"N final: {df['N_dunes'].iloc[-1]:.0f}")
print(f"W medio final: {df['mean_width'].iloc[-1]:.1f} m")
print(f"Colisiones totales: {df['collision_count'].iloc[-1]:.0f}")

In [ ]:
for dt_test in [0.6406, 0.3203, 0.1601, 0.0801]:
    model = DuneSwarm(**{**params_base, "dt": dt_test})
    n_steps = int(979 * 0.6406 / dt_test)  # mismo tiempo total
    for _ in range(n_steps):
        model.step()
    df = model.datacollector.get_model_vars_dataframe()
    print(f"dt={dt_test:.4f} | n_steps={n_steps:>5} | "
          f"N={df['N_dunes'].iloc[-1]:.0f} | "
          f"W={df['mean_width'].iloc[-1]:.1f}m | "
          f"colisiones={df['collision_count'].iloc[-1]:.0f}")

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path(".").resolve()))

from src.dune_swarm import DuneSwarm
import matplotlib.pyplot as plt

params_paper = {
    "qsat":         79.0,
    "q0ratio":      0.25,
    "qshift_ratio": 0.0,
    "lambda2_std":  0.0,
    "outflux_mode": "Hersen",
    "dt":           0.125,
    "w0":           16.6,
    "rho0":         37e-6,
    "simwidth":     9000.0,
    "simlength":    10000.0,
    "fieldwidth":   3000.0,
    "fieldlength":  10000.0,
    "inject":       True,
    "inject_mode":  "weq",
    "n_dunes_init": 0,
    "collisions":   True,
    "seed":         42,
    "lambda1":      1.0,
    "lambda2_mean": 1.8,
    "lambda3":      1/3,
    "alpha":        0.05,
    "delta":        4.6,
    "c":            45.0,
    "wind_regime":  "unimodal",
}

# n_steps = 500 años / dt = 4000 pasos
# Pero primero corremos 100 años (800 pasos) para ver si va en la dirección correcta
N_YEARS  = 100
n_steps  = int(N_YEARS / params_paper["dt"])

print(f"Corriendo {N_YEARS} años = {n_steps} pasos...")
print(f"W_eq = {4.6*79/(0.25*79 - 0.05*79):.1f} m  (paper ~23m)")
print(f"n_inject esperado/paso = {37e-6 * (45*79/(23+16.6)) * 0.125 * 3000:.2f}")

model = DuneSwarm(**params_paper)

checkpoints = set([1, 50, 100, 200, 400, 800])
for i in range(1, n_steps + 1):
    model.step()
    if i in checkpoints:
        agents = list(model.agents)
        if agents:
            ws = [a.width for a in agents]
            ys = [a.pos[1] for a in agents]
            print(f"  Paso {i:>4} ({i*params_paper['dt']:>5.1f} años) | "
                  f"N={len(agents):>4} | "
                  f"W_mean={sum(ws)/len(ws):>6.1f}m | "
                  f"y_min={min(ys):>7.0f}m")

df = model.datacollector.get_model_vars_dataframe()

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

# Convertir pasos a años
years = df.index * params_paper["dt"]

df["N_dunes"].values
axes[0].plot(years, df["N_dunes"], color="tab:orange", linewidth=1.5, label="N dunas")
axes[0].set_ylabel("N dunas")
axes[0].set_title("Validación paper Fig.1a — qshift=0, outflux=Hersen, ρ₀=37km⁻²")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(years, df["mean_width"], color="tab:blue", linewidth=1.5, label="Ancho medio (m)")
axes[1].set_ylabel("Ancho medio (m)")
axes[1].set_xlabel("Tiempo (años)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal {N_YEARS} años:")
print(f"  N_dunes    = {df['N_dunes'].iloc[-1]:.0f}  (paper Fig.1a ~800 en 500 años)")
print(f"  mean_width = {df['mean_width'].iloc[-1]:.1f}m  (paper Fig.1a ~80m en 500 años)")

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path(".").resolve()))

from src.dune_swarm import DuneSwarm
from src.flux_physics import flank_volume, horn_width

# ── Parámetros del paper ──────────────────────────────────────────────────────
lambda2 = 1.8
lambda3 = 1/3
alpha   = 0.05
delta   = 4.6
qsat    = 79.0
q0      = 0.25 * qsat   # = 19.75 m²/año
dt      = 0.125
w0      = 16.6
c       = 45.0
W_eq    = 23.0  # W de equilibrio para estos parámetros

# ── Crear modelo con UNA sola duna simétrica, sin inyección, sin colisiones ───
model = DuneSwarm(
    qsat=qsat, q0ratio=0.25, qshift_ratio=0.0,
    lambda2_std=0.0, wind_regime="unimodal",
    dt=dt, w0=w0, c=c,
    simwidth=500.0, simlength=500.0,
    fieldwidth=400.0, fieldlength=400.0,
    inject=False, n_dunes_init=0, collisions=False,
    outflux_mode="Hersen", seed=42,
    lambda1=1.0, lambda2_mean=lambda2, lambda3=lambda3,
    alpha=alpha, delta=delta,
)

# Insertar manualmente una duna simétrica de ancho W_eq
from src.dune_agent import DuneAgent
agent = DuneAgent(model, W_eq/2, W_eq/2)
model.space.place_agent(agent, (250.0, 250.0))
model.agents.add(agent)

# ── Calcular valores esperados analíticamente ─────────────────────────────────
lw = rw = W_eq / 2
V_inicial = 2 * flank_volume(lw, lambda2, lambda3)
H         = horn_width(lw, alpha, delta)

influx_esperado  = q0   * lw * dt   # q0 * W_proj * dt (cada flanco)
outflux_esperado = qsat * H  * dt   # qsat * H * dt (cada flanco)
dV_esperado      = 2 * (influx_esperado - outflux_esperado)  # ambos flancos

print("=" * 55)
print("  VALIDACIÓN UNITARIA — 1 duna simétrica W=W_eq")
print("=" * 55)
print(f"  W_eq          = {W_eq:.2f} m")
print(f"  lw = rw       = {lw:.2f} m")
print(f"  H (cuerno)    = {H:.3f} m")
print(f"  V_inicial     = {V_inicial:.4f} m³")
print(f"\n  --- Esperado analítico (paper Ec. 2) ---")
print(f"  influx/flanco = q0*lw*dt     = {influx_esperado:.6f} m³")
print(f"  outflux/flanco= qsat*H*dt    = {outflux_esperado:.6f} m³")
print(f"  dV total      = {dV_esperado:.6f} m³")
print(f"  dV/V_inicial  = {dV_esperado/V_inicial*100:.4f}%")
print(f"\n  --- En W_eq el balance debería ser ~0 ---")
print(f"  influx == outflux? {abs(influx_esperado - outflux_esperado) < 1e-6}")

# ── Correr 1 paso y medir lo que realmente ocurre ────────────────────────────
V_antes = 2 * flank_volume(agent.lw, lambda2, lambda3)

# Guardar flujos antes del step
model._wind_vec = (0.0, -1.0)
from src.flux_physics import distribute_flux
distribute_flux(
    agents=[agent],
    wind_vec=model._wind_vec,
    qsat=model.qsat,
    q0=model.q0,
    dt=model.dt,
    w0=model.w0,
    c=model.c,
    alpha=model.alpha,
    delta=model.delta,
    outflux_mode=model.outflux_mode,
)

influx_real_l  = agent._influx_l
influx_real_r  = agent._influx_r
outflux_real_l = agent._outflux_l
outflux_real_r = agent._outflux_r

# Ejecutar el paso completo
model.step()

V_despues = 2 * flank_volume(agent.lw, lambda2, lambda3)
dV_real   = V_despues - V_antes

print(f"\n  --- Real (tu implementación) ---")
print(f"  influx  izq   = {influx_real_l:.6f} m³")
print(f"  influx  der   = {influx_real_r:.6f} m³")
print(f"  outflux izq   = {outflux_real_l:.6f} m³")
print(f"  outflux der   = {outflux_real_r:.6f} m³")
print(f"  dV real       = {dV_real:.6f} m³")
print(f"  dV/V_inicial  = {dV_real/V_antes*100:.4f}%")

print(f"\n  --- Comparación ---")
print(f"  Error influx  = {(influx_real_l - influx_esperado)/influx_esperado*100:.2f}%")
print(f"  Error outflux = {(outflux_real_l - outflux_esperado)/outflux_esperado*100:.2f}%")
print(f"  Error dV      = {abs(dV_real - dV_esperado):.6f} m³")

In [ ]:
# Después del distribute_flux pero ANTES del step
# ya lo capturamos arriba — el outflux_real_l = 28.39 ✓

# El problema debe estar en schedule_outflux vs _outflux_l
# Verifica qué valor tiene _outflux_l justo antes de _update_volumes

import src.dune_agent as da
original_update = da.DuneAgent._update_volumes

def patched_update(self):
    print(f"  [_update_volumes] lw={self.lw:.3f} rw={self.rw:.3f}")
    print(f"    _influx_l={self._influx_l:.4f}  _outflux_l={self._outflux_l:.4f}")
    print(f"    _influx_r={self._influx_r:.4f}  _outflux_r={self._outflux_r:.4f}")
    original_update(self)
    print(f"  [post] _vl_new={self._vl_new:.4f}  _vr_new={self._vr_new:.4f}")

da.DuneAgent._update_volumes = patched_update

# Nueva duna y modelo limpio
model2 = DuneSwarm(
    qsat=79.0, q0ratio=0.25, qshift_ratio=0.0,
    lambda2_std=0.0, wind_regime="unimodal",
    dt=0.125, w0=16.6, c=45.0,
    simwidth=500.0, simlength=500.0,
    fieldwidth=400.0, fieldlength=400.0,
    inject=False, n_dunes_init=0, collisions=False,
    outflux_mode="Hersen", seed=42,
    lambda1=1.0, lambda2_mean=1.8, lambda3=1/3,
    alpha=0.05, delta=4.6,
)
agent2 = DuneAgent(model2, 11.5, 11.5)
model2.space.place_agent(agent2, (250.0, 250.0))
model2.agents.add(agent2)
model2.step()

da.DuneAgent._update_volumes = original_update

In [ ]:
import src.dune_agent as da
original_receive = da.DuneAgent.receive_flux

call_count = [0]

def patched_receive(self, flux_l, flux_r):
    call_count[0] += 1
    print(f"  [receive_flux #{call_count[0]}] flux_l={flux_l:.4f}  flux_r={flux_r:.4f}")
    import traceback
    traceback.print_stack(limit=4)
    original_receive(self, flux_l, flux_r)

da.DuneAgent.receive_flux = patched_receive

model3 = DuneSwarm(
    qsat=79.0, q0ratio=0.25, qshift_ratio=0.0,
    lambda2_std=0.0, wind_regime="unimodal",
    dt=0.125, w0=16.6, c=45.0,
    simwidth=500.0, simlength=500.0,
    fieldwidth=400.0, fieldlength=400.0,
    inject=False, n_dunes_init=0, collisions=False,
    outflux_mode="Hersen", seed=42,
    lambda1=1.0, lambda2_mean=1.8, lambda3=1/3,
    alpha=0.05, delta=4.6,
)
agent3 = DuneAgent(model3, 11.5, 11.5)
model3.space.place_agent(agent3, (250.0, 250.0))
model3.agents.add(agent3)
model3.step()

da.DuneAgent.receive_flux = original_receive

In [ ]:
import src.dune_agent as da
original_receive = da.DuneAgent.receive_flux

calls = []

def patched_receive(self, flux_l, flux_r):
    calls.append((flux_l, flux_r))
    original_receive(self, flux_l, flux_r)

da.DuneAgent.receive_flux = patched_receive

model3 = DuneSwarm(
    qsat=79.0, q0ratio=0.25, qshift_ratio=0.0,
    lambda2_std=0.0, wind_regime="unimodal",
    dt=0.125, w0=16.6, c=45.0,
    simwidth=500.0, simlength=500.0,
    fieldwidth=400.0, fieldlength=400.0,
    inject=False, n_dunes_init=0, collisions=False,
    outflux_mode="Hersen", seed=42,
    lambda1=1.0, lambda2_mean=1.8, lambda3=1/3,
    alpha=0.05, delta=4.6,
)
agent3 = DuneAgent(model3, 11.5, 11.5)
model3.space.place_agent(agent3, (250.0, 250.0))
model3.agents.add(agent3)
model3.step()

da.DuneAgent.receive_flux = original_receive

print(f"receive_flux llamado {len(calls)} veces:")
for i, (l, r) in enumerate(calls):
    print(f"  #{i+1}: flux_l={l:.4f}  flux_r={r:.4f}")
print(f"Total influx_l = {sum(c[0] for c in calls):.4f}")
print(f"Total influx_r = {sum(c[1] for c in calls):.4f}")

In [ ]:
import src.dune_agent as da
original_update = da.DuneAgent._update_volumes

update_calls = []

def patched_update(self):
    update_calls.append((self._influx_l, self._outflux_l, self.lw))
    original_update(self)

da.DuneAgent._update_volumes = patched_update

model4 = DuneSwarm(
    qsat=79.0, q0ratio=0.25, qshift_ratio=0.0,
    lambda2_std=0.0, wind_regime="unimodal",
    dt=0.125, w0=16.6, c=45.0,
    simwidth=500.0, simlength=500.0,
    fieldwidth=400.0, fieldlength=400.0,
    inject=False, n_dunes_init=0, collisions=False,
    outflux_mode="Hersen", seed=42,
    lambda1=1.0, lambda2_mean=1.8, lambda3=1/3,
    alpha=0.05, delta=4.6,
)
agent4 = DuneAgent(model4, 11.5, 11.5)
model4.space.place_agent(agent4, (250.0, 250.0))
model4.agents.add(agent4)

# Correr DOS pasos para ver si acumula
for paso in range(2):
    update_calls.clear()
    model4.step()
    print(f"\nPaso {paso+1}: _update_volumes llamado {len(update_calls)} veces")
    for i, (inf, out, lw) in enumerate(update_calls):
        print(f"  #{i+1}: influx={inf:.4f}  outflux={out:.4f}  lw_antes={lw:.4f}")
    agents = list(model4.agents)
    if agents:
        a = agents[0]
        print(f"  lw_final={a.lw:.4f}  V_final={2*flank_volume(a.lw,1.8,1/3):.4f}")

da.DuneAgent._update_volumes = original_update

In [ ]:
model5 = DuneSwarm(
    qsat=79.0, q0ratio=0.25, qshift_ratio=0.0,
    lambda2_std=0.0, wind_regime="unimodal",
    dt=0.125, w0=16.6, c=45.0,
    simwidth=500.0, simlength=500.0,
    fieldwidth=400.0, fieldlength=400.0,
    inject=False, n_dunes_init=0, collisions=False,
    outflux_mode="Hersen", seed=42,
    lambda1=1.0, lambda2_mean=1.8, lambda3=1/3,
    alpha=0.05, delta=4.6,
)
agent5 = DuneAgent(model5, 11.5, 11.5)
model5.space.place_agent(agent5, (250.0, 250.0))
model5.agents.add(agent5)

print("Evolución de V a lo largo de 10 pasos (W=W_eq, debe ser estable):")
for i in range(10):
    V = 2 * flank_volume(agent5.lw, 1.8, 1/3)
    model5.step()
    V_new = 2 * flank_volume(agent5.lw, 1.8, 1/3)
    print(f"  Paso {i+1:>2}: V={V:.4f} → {V_new:.4f}  dV={V_new-V:+.6f}")

In [ ]:
from src.flux_physics import projected_width, horn_width
from src.wind_regimes import get_angle
import math

wind_vec   = (0.0, -1.0)
wind_angle = get_angle(wind_vec)
lw         = 11.5
alpha, delta, qsat, q0, dt = 0.05, 4.6, 79.0, 19.75, 0.125

proj  = projected_width(lw, wind_angle)
H     = horn_width(lw, alpha, delta)

influx_real  = q0   * proj * dt
outflux_real = qsat * H    * dt

print(f"wind_angle     = {wind_angle:.6f} rad  ({math.degrees(wind_angle):.2f}°)")
print(f"|sin(angle)|   = {abs(math.sin(wind_angle)):.10f}")
print(f"proj (lw*|sin|)= {proj:.10f}  vs lw = {lw:.10f}")
print(f"H (cuerno)     = {H:.10f}")
print(f"influx         = {influx_real:.10f}")
print(f"outflux        = {outflux_real:.10f}")
print(f"diferencia     = {influx_real - outflux_real:.2e}")
print(f"")
print(f"W_eq exacto    = delta*qsat/(q0 - alpha*qsat)")
W_eq_exact = delta * qsat / (q0 - alpha * qsat)
print(f"               = {W_eq_exact:.10f} m  (usaste {lw*2:.1f}m)")

In [ ]:
model6 = DuneSwarm(
    qsat=79.0, q0ratio=0.25, qshift_ratio=0.0,
    lambda2_std=0.0, wind_regime="unimodal",
    dt=0.125, w0=16.6, c=45.0,
    simwidth=500.0, simlength=500.0,
    fieldwidth=400.0, fieldlength=400.0,
    inject=False, n_dunes_init=0, collisions=False,
    outflux_mode="Hersen", seed=42,
    lambda1=1.0, lambda2_mean=1.8, lambda3=1/3,
    alpha=0.05, delta=4.6,
)
agent6 = DuneAgent(model6, 11.5, 11.5)
model6.space.place_agent(agent6, (250.0, 250.0))
model6.agents.add(agent6)

print("Paso | lw      | rw      | asym   | V       | N_agents | calving")
print("-"*70)
for i in range(20):
    V    = 2 * flank_volume(agent6.lw, 1.8, 1/3) if agent6 in model6.agents else 0
    lw   = agent6.lw if agent6 in model6.agents else 0
    rw   = agent6.rw if agent6 in model6.agents else 0
    asym = agent6.asymmetry if agent6 in model6.agents else 0
    cal_antes = model6.calving_count

    model6.step()

    n    = len(list(model6.agents))
    cal  = model6.calving_count - cal_antes
    print(f"  {i+1:>2} | {lw:>7.4f} | {rw:>7.4f} | {asym:.4f} | {V:>8.4f} | {n:>8} | {cal}")

In [ ]:
from src.flux_physics import flank_volume, width_from_volume

lambda2, lambda3 = 1.8, 1/3
lw = 11.5

# Ciclo: lw → V → lw_reconstruido → V_reconstruido
V_original      = flank_volume(lw, lambda2, lambda3)
lw_reconstruido = width_from_volume(V_original, lambda2, lambda3)
V_reconstruido  = flank_volume(lw_reconstruido, lambda2, lambda3)

print(f"lw original      = {lw:.10f}")
print(f"V original       = {V_original:.10f}")
print(f"lw reconstruido  = {lw_reconstruido:.10f}")
print(f"V reconstruido   = {V_reconstruido:.10f}")
print(f"Error roundtrip  = {V_reconstruido - V_original:.2e}")

# Ahora simular lo que hace _update_volumes exactamente
influx  = 28.390625
outflux = 28.390625
V_new   = V_original + influx - outflux   # debe ser V_original
lw_new  = width_from_volume(V_new, lambda2, lambda3)
V_check = flank_volume(lw_new, lambda2, lambda3)

print(f"\nDespués de _update_volumes + _recalc_widths:")
print(f"V_new (antes recalc) = {V_new:.10f}")
print(f"lw_new               = {lw_new:.10f}")
print(f"V_check (roundtrip)  = {V_check:.10f}")
print(f"Pérdida por paso     = {V_check - V_original:.2e}")

In [ ]:
from src.wind_regimes import WindRegime, get_angle
import numpy as np
import math

rng = np.random.default_rng(42)
regime = WindRegime(
    regime="unimodal", mean_deg=270.0, std_deg=3.0,
    rng=rng
)

lw = 11.5
alpha, delta = 0.05, 4.6
qsat, q0, dt = 79.0, 19.75, 0.125
H = alpha * lw + delta/2

losses = []
for _ in range(100):
    wd = regime.sample()
    angle = get_angle(wd)
    proj  = lw * abs(math.sin(angle))
    influx  = q0   * proj * dt
    outflux = qsat * H    * dt
    losses.append(influx - outflux)

print(f"dV medio por flanco por paso = {np.mean(losses):.6f} m³")
print(f"dV total por paso (2 flancos)= {2*np.mean(losses):.6f} m³")
print(f"W_eq con viento puro 270°    = 23.00 m  (sin dispersión)")
print(f"\nCon std_deg=3°, el influx efectivo es:")
angles = [get_angle(regime.sample()) for _ in range(10000)]
mean_sin = np.mean([abs(math.sin(a)) for a in angles])
print(f"  E[|sin(angle)|] = {mean_sin:.6f}  (vs 1.0 para viento puro)")
print(f"  W_eq efectivo   = {delta*qsat/(q0*mean_sin - alpha*qsat):.2f} m")

In [ ]:
model7 = DuneSwarm(
    qsat=79.0, q0ratio=0.25, qshift_ratio=0.0,
    lambda2_std=0.0, wind_regime="unimodal",
    dt=0.125, w0=16.6, c=45.0,
    simwidth=500.0, simlength=1000.0,
    fieldwidth=400.0, fieldlength=900.0,
    inject=False, n_dunes_init=0, collisions=False,
    outflux_mode="Hersen", seed=42,
    lambda1=1.0, lambda2_mean=1.8, lambda3=1/3,
    alpha=0.05, delta=4.6,
)

# Duna 1 barlovento (y alto), duna 2 sotavento (y bajo), alineadas en x
a1 = DuneAgent(model7, 11.5, 11.5)
a2 = DuneAgent(model7, 11.5, 11.5)
model7.space.place_agent(a1, (250.0, 700.0))
model7.space.place_agent(a2, (250.0, 300.0))
model7.agents.add(a1)
model7.agents.add(a2)

print("Paso | V_total  | V_a1     | V_a2     | dV_total")
print("-"*60)
for i in range(10):
    V1 = 2 * flank_volume(a1.lw, 1.8, 1/3) if a1 in model7.agents else 0
    V2 = 2 * flank_volume(a2.lw, 1.8, 1/3) if a2 in model7.agents else 0
    Vtot = V1 + V2
    model7.step()
    V1n = 2 * flank_volume(a1.lw, 1.8, 1/3) if a1 in model7.agents else 0
    V2n = 2 * flank_volume(a2.lw, 1.8, 1/3) if a2 in model7.agents else 0
    Vtotn = V1n + V2n
    print(f"  {i+1:>2} | {Vtot:>8.3f} | {V1:>8.3f} | {V2:>8.3f} | {Vtotn-Vtot:>+.4f}")

In [ ]:
import src.dune_agent as da
original_receive = da.DuneAgent.receive_flux

calls_a1 = []
calls_a2 = []

def patched_receive(self, flux_l, flux_r):
    if self is agent_bar:
        calls_a1.append((flux_l, flux_r))
    elif self is agent_sot:
        calls_a2.append((flux_l, flux_r))
    original_receive(self, flux_l, flux_r)

da.DuneAgent.receive_flux = patched_receive

model8 = DuneSwarm(
    qsat=79.0, q0ratio=0.25, qshift_ratio=0.0,
    lambda2_std=0.0, wind_regime="unimodal",
    dt=0.125, w0=16.6, c=45.0,
    simwidth=500.0, simlength=1000.0,
    fieldwidth=400.0, fieldlength=900.0,
    inject=False, n_dunes_init=0, collisions=False,
    outflux_mode="Hersen", seed=42,
    lambda1=1.0, lambda2_mean=1.8, lambda3=1/3,
    alpha=0.05, delta=4.6,
)

agent_bar = DuneAgent(model8, 11.5, 11.5)
agent_sot = DuneAgent(model8, 11.5, 11.5)
model8.space.place_agent(agent_bar, (250.0, 700.0))
model8.space.place_agent(agent_sot, (250.0, 300.0))
model8.agents.add(agent_bar)
model8.agents.add(agent_sot)

model8.step()

da.DuneAgent.receive_flux = original_receive

print(f"Duna barlovento (a1) — receive_flux llamado {len(calls_a1)} veces:")
for i, (l, r) in enumerate(calls_a1):
    print(f"  #{i+1}: flux_l={l:.4f}  flux_r={r:.4f}  total={l+r:.4f}")
print(f"  TOTAL influx a1 = {sum(c[0]+c[1] for c in calls_a1):.4f}")

print(f"\nDuna sotavento (a2) — receive_flux llamado {len(calls_a2)} veces:")
for i, (l, r) in enumerate(calls_a2):
    print(f"  #{i+1}: flux_l={l:.4f}  flux_r={r:.4f}  total={l+r:.4f}")
print(f"  TOTAL influx a2 = {sum(c[0]+c[1] for c in calls_a2):.4f}")

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path(".").resolve()))

# Reimportar modulos frescos
import importlib
import src.flux_physics
importlib.reload(src.flux_physics)
import src.dune_agent
importlib.reload(src.dune_agent)
import src.dune_swarm
importlib.reload(src.dune_swarm)

from src.dune_swarm import DuneSwarm
from src.dune_agent import DuneAgent
from src.flux_physics import flank_volume

model = DuneSwarm(
    qsat=79.0, q0ratio=0.25, qshift_ratio=0.0,
    lambda2_std=0.0, wind_regime="unimodal",
    dt=0.125, w0=16.6, c=45.0,
    simwidth=500.0, simlength=1000.0,
    fieldwidth=400.0, fieldlength=900.0,
    inject=False, n_dunes_init=0, collisions=False,
    outflux_mode="Hersen", seed=42,
    lambda1=1.0, lambda2_mean=1.8, lambda3=1/3,
    alpha=0.05, delta=4.6,
)

a1 = DuneAgent(model, 11.5, 11.5)
a2 = DuneAgent(model, 11.5, 11.5)
model.space.place_agent(a1, (250.0, 700.0))
model.space.place_agent(a2, (250.0, 300.0))
model.agents.add(a1)
model.agents.add(a2)

print("Paso | V_total  | V_a1     | V_a2     | dV_total")
print("-"*60)
for i in range(10):
    V1   = 2 * flank_volume(a1.lw, 1.8, 1/3) if a1 in model.agents else 0
    V2   = 2 * flank_volume(a2.lw, 1.8, 1/3) if a2 in model.agents else 0
    Vtot = V1 + V2
    model.step()
    V1n   = 2 * flank_volume(a1.lw, 1.8, 1/3) if a1 in model.agents else 0
    V2n   = 2 * flank_volume(a2.lw, 1.8, 1/3) if a2 in model.agents else 0
    Vtotn = V1n + V2n
    print(f"  {i+1:>2} | {Vtot:>8.3f} | {V1:>8.3f} | {V2:>8.3f} | {Vtotn-Vtot:>+.5f}")

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path(".").resolve()))

from src.dune_swarm import DuneSwarm
from src.dune_agent import DuneAgent
from src.flux_physics import flank_volume

model = DuneSwarm(
    qsat=79.0, q0ratio=0.25, qshift_ratio=0.0,
    lambda2_std=0.0, wind_regime="unimodal",
    dt=0.125, w0=16.6, c=45.0,
    simwidth=500.0, simlength=1000.0,
    fieldwidth=400.0, fieldlength=900.0,
    inject=False, n_dunes_init=0, collisions=False,
    outflux_mode="Hersen", seed=42,
    lambda1=1.0, lambda2_mean=1.8, lambda3=1/3,
    alpha=0.05, delta=4.6,
)

a1 = DuneAgent(model, 11.5, 11.5)
a2 = DuneAgent(model, 11.5, 11.5)
model.space.place_agent(a1, (250.0, 700.0))
model.space.place_agent(a2, (250.0, 300.0))
model.agents.add(a1)
model.agents.add(a2)

print("Paso | V_total  | V_a1     | V_a2     | dV_total")
print("-"*60)
for i in range(10):
    V1   = 2 * flank_volume(a1.lw, 1.8, 1/3) if a1 in model.agents else 0
    V2   = 2 * flank_volume(a2.lw, 1.8, 1/3) if a2 in model.agents else 0
    Vtot = V1 + V2
    model.step()
    V1n   = 2 * flank_volume(a1.lw, 1.8, 1/3) if a1 in model.agents else 0
    V2n   = 2 * flank_volume(a2.lw, 1.8, 1/3) if a2 in model.agents else 0
    Vtotn = V1n + V2n
    print(f"  {i+1:>2} | {Vtot:>8.3f} | {V1:>8.3f} | {V2:>8.3f} | {Vtotn-Vtot:>+.5f}")

In [ ]:
import src.flux_physics
print(src.flux_physics.__file__)

# Ver las líneas alrededor del fix
import inspect
src_code = inspect.getsource(src.flux_physics.distribute_flux)
# Buscar la línea del fix
for i, line in enumerate(src_code.split('\n')):
    if 'Nunca' in line or 'other is agent' in line:
        print(f"Línea {i}: {line}")

In [ ]:
import src.dune_agent as da
original_receive = da.DuneAgent.receive_flux

calls_a1 = []
calls_a2 = []

def patched_receive(self, flux_l, flux_r):
    if self is a1:
        calls_a1.append((flux_l, flux_r))
    elif self is a2:
        calls_a2.append((flux_l, flux_r))
    original_receive(self, flux_l, flux_r)

da.DuneAgent.receive_flux = patched_receive

# Un solo paso limpio
model2 = DuneSwarm(
    qsat=79.0, q0ratio=0.25, qshift_ratio=0.0,
    lambda2_std=0.0, wind_regime="unimodal",
    dt=0.125, w0=16.6, c=45.0,
    simwidth=500.0, simlength=1000.0,
    fieldwidth=400.0, fieldlength=900.0,
    inject=False, n_dunes_init=0, collisions=False,
    outflux_mode="Hersen", seed=42,
    lambda1=1.0, lambda2_mean=1.8, lambda3=1/3,
    alpha=0.05, delta=4.6,
)
a1 = DuneAgent(model2, 11.5, 11.5)
a2 = DuneAgent(model2, 11.5, 11.5)
model2.space.place_agent(a1, (250.0, 700.0))
model2.space.place_agent(a2, (250.0, 300.0))
model2.agents.add(a1)
model2.agents.add(a2)

model2.step()

da.DuneAgent.receive_flux = original_receive

print(f"a1 (barlovento) — llamadas: {len(calls_a1)}")
for i,(l,r) in enumerate(calls_a1):
    print(f"  #{i+1}: l={l:.4f} r={r:.4f} total={l+r:.4f}")

print(f"\na2 (sotavento) — llamadas: {len(calls_a2)}")
for i,(l,r) in enumerate(calls_a2):
    print(f"  #{i+1}: l={l:.4f} r={r:.4f} total={l+r:.4f}")

In [ ]:
model3 = DuneSwarm(
    qsat=79.0, q0ratio=0.25, qshift_ratio=0.0,
    lambda2_std=0.0, wind_regime="unimodal",
    dt=0.125, w0=16.6, c=45.0,
    simwidth=500.0, simlength=1000.0,
    fieldwidth=400.0, fieldlength=900.0,
    inject=False, n_dunes_init=0, collisions=False,
    outflux_mode="Hersen", seed=42,
    lambda1=1.0, lambda2_mean=1.8, lambda3=1/3,
    alpha=0.05, delta=4.6,
)
a1 = DuneAgent(model3, 11.5, 11.5)
a2 = DuneAgent(model3, 11.5, 11.5)
model3.space.place_agent(a1, (250.0, 700.0))
model3.space.place_agent(a2, (250.0, 300.0))
model3.agents.add(a1)
model3.agents.add(a2)

# Verificar identidad dentro del loop de distribute_flux
active = list(model3.agents)
print(f"a1 id = {id(a1)}")
print(f"a2 id = {id(a2)}")
print(f"Agentes en active:")
for ag in active:
    print(f"  id={id(ag)}  is a1={ag is a1}  is a2={ag is a2}")

In [ ]:
import math
from src.wind_regimes import get_angle

wind_vec   = (0.0, -1.0)
wx, wy     = wind_vec

def upwind_projection(pos):
    x, y = pos
    return -(x * wx + y * wy)

pos_a1 = (250.0, 700.0)
pos_a2 = (250.0, 300.0)

up1 = upwind_projection(pos_a1)
up2 = upwind_projection(pos_a2)

print(f"upwind_projection(a1, y=700) = {up1}")
print(f"upwind_projection(a2, y=300) = {up2}")
print(f"a1 más barlovento que a2: {up1 > up2}  ← debe ser True")
print(f"sorted order (mayor primero): a1 primero = {up1 > up2}")
print()
print(f"Con wind=[0,-1]: -(x*0 + y*(-1)) = y")
print(f"  a1: upwind = {up1}  (y=700)")
print(f"  a2: upwind = {up2}  (y=300)")
print(f"  sorted descendente: a1({up1}) > a2({up2}) → a1 primero ✓")
print()

# Verificar la condición del loop
# Cuando procesamos a1 (agent), buscamos others con proyección MENOR
print(f"Cuando procesamos a1 (proj={up1}):")
print(f"  a2 proj={up2} <= {up1}? {up2 <= up1} → {'SKIP (correcto)' if up2 <= up1 else 'PROPAGAR (error)'}")
print()
print(f"Cuando procesamos a2 (proj={up2}):")
print(f"  a1 proj={up1} <= {up2}? {up1 <= up2} → {'SKIP (correcto)' if up1 <= up2 else 'PROPAGAR (error)'}")

In [ ]:
import src.flux_physics, inspect
src_code = inspect.getsource(src.flux_physics.distribute_flux)
has_shapely = "_SHAPELY" in src_code
has_shadow  = "_shadow" in src_code
has_overlap = "_overlap_width" in src_code
print(f"Shapely implementado : {has_shapely}")
print(f"Sombra de flujo      : {has_shadow}")
print(f"Overlap geométrico   : {has_overlap}")
print(f"Fix completo         : {all([has_shapely, has_shadow, has_overlap])}")

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path(".").resolve()))

from src.dune_swarm import DuneSwarm
from src.dune_agent import DuneAgent
from src.flux_physics import flank_volume

# Verificar que el fix está activo
import src.flux_physics, inspect
src_code = inspect.getsource(src.flux_physics.distribute_flux)
has_fix = "upwind_projection(other) >= upwind_projection(agent)" in src_code
print(f"Fix aplicado: {has_fix}")
print()

model = DuneSwarm(
    qsat=79.0, q0ratio=0.25, qshift_ratio=0.0,
    lambda2_std=0.0, wind_regime="unimodal",
    dt=0.125, w0=16.6, c=45.0,
    simwidth=500.0, simlength=1000.0,
    fieldwidth=400.0, fieldlength=900.0,
    inject=False, n_dunes_init=0, collisions=False,
    outflux_mode="Hersen", seed=42,
    lambda1=1.0, lambda2_mean=1.8, lambda3=1/3,
    alpha=0.05, delta=4.6,
)

a1 = DuneAgent(model, 11.5, 11.5)
a2 = DuneAgent(model, 11.5, 11.5)
model.space.place_agent(a1, (250.0, 700.0))
model.space.place_agent(a2, (250.0, 300.0))
model.agents.add(a1)
model.agents.add(a2)

print("Paso | V_total  | V_a1     | V_a2     | dV_total")
print("-"*60)
for i in range(10):
    V1   = 2 * flank_volume(a1.lw, 1.8, 1/3) if a1 in model.agents else 0
    V2   = 2 * flank_volume(a2.lw, 1.8, 1/3) if a2 in model.agents else 0
    Vtot = V1 + V2
    model.step()
    V1n   = 2 * flank_volume(a1.lw, 1.8, 1/3) if a1 in model.agents else 0
    V2n   = 2 * flank_volume(a2.lw, 1.8, 1/3) if a2 in model.agents else 0
    Vtotn = V1n + V2n
    print(f"  {i+1:>2} | {Vtot:>8.3f} | {V1:>8.3f} | {V2:>8.3f} | {Vtotn-Vtot:>+.6f}")

In [ ]:
model_1duna = DuneSwarm(
    qsat=79.0, q0ratio=0.25, qshift_ratio=0.0,
    lambda2_std=0.0, wind_regime="unimodal",
    dt=0.125, w0=16.6, c=45.0,
    simwidth=500.0, simlength=500.0,
    fieldwidth=400.0, fieldlength=400.0,
    inject=False, n_dunes_init=0, collisions=False,
    outflux_mode="Hersen", seed=42,
    lambda1=1.0, lambda2_mean=1.8, lambda3=1/3,
    alpha=0.05, delta=4.6,
)

a = DuneAgent(model_1duna, 11.5, 11.5)
model_1duna.space.place_agent(a, (250.0, 250.0))
model_1duna.agents.add(a)

print("Paso | V        | dV")
print("-"*35)
for i in range(10):
    V = 2 * flank_volume(a.lw, 1.8, 1/3)
    model_1duna.step()
    Vn = 2 * flank_volume(a.lw, 1.8, 1/3)
    print(f"  {i+1:>2} | {V:>8.4f} | {Vn-V:>+.6f}")

In [ ]:
import src.flux_physics as fp
import inspect

# Ver qué devuelve _overlap_width para la duna en el campo de flujo
from shapely.geometry import box as shp_box
from src.flux_physics import _barchan_poly, _rotate_poly, _shadow, _overlap_width

wind_vec = (0.0, -1.0)
x, y, lw, rw = 250.0, 250.0, 11.5, 11.5
lambda1, lambda2, alpha, delta = 1.0, 1.8, 0.05, 4.6
q0, dt = 19.75, 0.125

# Campo de flujo del tamaño del dominio
fluxfield = shp_box(-1e7, -1e7, 1e7, 1e7)
fluxfield_rot = _rotate_poly(fluxfield, wind_vec)

left, right = _barchan_poly(x, y, lw, rw, lambda1, lambda2, alpha, delta)
left_rot  = _rotate_poly(left,  wind_vec)
right_rot = _rotate_poly(right, wind_vec)

ff_after = _shadow(fluxfield_rot, left_rot, right_rot, wind_vec)

ow_l = _overlap_width(ff_after, left_rot)
ow_r = _overlap_width(ff_after, right_rot)

influx_l = ow_l * q0 * dt
influx_r = ow_r * q0 * dt

print(f"overlap_width izq = {ow_l:.6f} m  (esperado: {lw:.6f} m)")
print(f"overlap_width der = {ow_r:.6f} m  (esperado: {rw:.6f} m)")
print(f"influx_l          = {influx_l:.6f} m³ (esperado: {q0*lw*dt:.6f} m³)")
print(f"influx_r          = {influx_r:.6f} m³ (esperado: {q0*rw*dt:.6f} m³)")
print(f"Error influx_l    = {(influx_l - q0*lw*dt):.6f} m³")

In [ ]:
model_fijo = DuneSwarm(
    qsat=79.0, q0ratio=0.25, qshift_ratio=0.0,
    lambda2_std=0.0, wind_regime="unimodal",
    wind_std_deg=0.0,  # ← viento perfectamente fijo
    dt=0.125, w0=16.6, c=45.0,
    simwidth=500.0, simlength=500.0,
    fieldwidth=400.0, fieldlength=400.0,
    inject=False, n_dunes_init=0, collisions=False,
    outflux_mode="Hersen", seed=42,
    lambda1=1.0, lambda2_mean=1.8, lambda3=1/3,
    alpha=0.05, delta=4.6,
)

a = DuneAgent(model_fijo, 11.5, 11.5)
model_fijo.space.place_agent(a, (250.0, 250.0))
model_fijo.agents.add(a)

print("Paso | V        | dV")
print("-"*35)
for i in range(10):
    V = 2 * flank_volume(a.lw, 1.8, 1/3)
    model_fijo.step()
    Vn = 2 * flank_volume(a.lw, 1.8, 1/3)
    print(f"  {i+1:>2} | {V:>8.4f} | {Vn-V:>+.6f}")

In [ ]:
model_2 = DuneSwarm(
    qsat=79.0, q0ratio=0.25, qshift_ratio=0.0,
    lambda2_std=0.0, wind_regime="unimodal",
    wind_std_deg=0.0,  # viento fijo para validar conservación pura
    dt=0.125, w0=16.6, c=45.0,
    simwidth=500.0, simlength=1000.0,
    fieldwidth=400.0, fieldlength=900.0,
    inject=False, n_dunes_init=0, collisions=False,
    outflux_mode="Hersen", seed=42,
    lambda1=1.0, lambda2_mean=1.8, lambda3=1/3,
    alpha=0.05, delta=4.6,
)

a1 = DuneAgent(model_2, 11.5, 11.5)
a2 = DuneAgent(model_2, 11.5, 11.5)
model_2.space.place_agent(a1, (250.0, 700.0))
model_2.space.place_agent(a2, (250.0, 300.0))
model_2.agents.add(a1)
model_2.agents.add(a2)

print("Paso | V_total  | V_a1     | V_a2     | dV_total")
print("-"*60)
for i in range(10):
    V1   = 2 * flank_volume(a1.lw, 1.8, 1/3) if a1 in model_2.agents else 0
    V2   = 2 * flank_volume(a2.lw, 1.8, 1/3) if a2 in model_2.agents else 0
    Vtot = V1 + V2
    model_2.step()
    V1n   = 2 * flank_volume(a1.lw, 1.8, 1/3) if a1 in model_2.agents else 0
    V2n   = 2 * flank_volume(a2.lw, 1.8, 1/3) if a2 in model_2.agents else 0
    Vtotn = V1n + V2n
    print(f"  {i+1:>2} | {Vtot:>8.3f} | {V1:>8.3f} | {V2:>8.3f} | {Vtotn-Vtot:>+.6f}")

In [ ]:
model_3 = DuneSwarm(
    qsat=79.0, q0ratio=0.25, qshift_ratio=0.0,
    lambda2_std=0.0, wind_regime="unimodal",
    wind_std_deg=3.0,  # viento estocástico
    dt=0.125, w0=16.6, c=45.0,
    simwidth=500.0, simlength=1000.0,
    fieldwidth=400.0, fieldlength=900.0,
    inject=False, n_dunes_init=0, collisions=False,
    outflux_mode="Hersen", seed=42,
    lambda1=1.0, lambda2_mean=1.8, lambda3=1/3,
    alpha=0.05, delta=4.6,
)

a1 = DuneAgent(model_3, 11.5, 11.5)
a2 = DuneAgent(model_3, 11.5, 11.5)
model_3.space.place_agent(a1, (250.0, 700.0))
model_3.space.place_agent(a2, (250.0, 300.0))
model_3.agents.add(a1)
model_3.agents.add(a2)

dVs = []
print("Paso | V_total  | dV_total")
print("-"*40)
for i in range(50):
    V1   = 2 * flank_volume(a1.lw, 1.8, 1/3) if a1 in model_3.agents else 0
    V2   = 2 * flank_volume(a2.lw, 1.8, 1/3) if a2 in model_3.agents else 0
    Vtot = V1 + V2
    model_3.step()
    V1n   = 2 * flank_volume(a1.lw, 1.8, 1/3) if a1 in model_3.agents else 0
    V2n   = 2 * flank_volume(a2.lw, 1.8, 1/3) if a2 in model_3.agents else 0
    Vtotn = V1n + V2n
    dV = Vtotn - Vtot
    dVs.append(dV)
    if i < 10 or i % 10 == 9:
        print(f"  {i+1:>2} | {Vtot:>8.3f} | {dV:>+.4f}")

import numpy as np
print(f"\nEstadísticas dV sobre 50 pasos:")
print(f"  Media  = {np.mean(dVs):>+.4f} m³  ← debe ser ~0")
print(f"  Std    = {np.std(dVs):>.4f} m³")
print(f"  Máx    = {max(dVs):>+.4f} m³")
print(f"  Mín    = {min(dVs):>+.4f} m³")

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path(".").resolve()))

from src.dune_swarm import DuneSwarm
from src.dune_agent import DuneAgent
from src.flux_physics import flank_volume
import matplotlib.pyplot as plt
import numpy as np

# Verificar que el nuevo flux_physics está activo
import src.flux_physics, inspect
src_code = inspect.getsource(src.flux_physics.distribute_flux)
print(f"Shapely activo: {'_SHAPELY' in src_code}")
print(f"Sombra activa : {'_shadow' in src_code}")
print()

# ── Parámetros exactos del paper Fig. 1a ──────────────────────────────────────
# qshift=0, outflux=Hersen, rho0=37 km⁻², 500 años
params_1a = dict(
    qsat         = 79.0,
    q0ratio      = 0.25,
    qshift_ratio = 0.0,
    lambda2_std  = 0.0,
    outflux_mode = "Hersen",
    dt           = 0.125,
    w0           = 16.6,
    c            = 45.0,
    rho0         = 37e-6,       # 37 km⁻² en m⁻²
    inject       = True,
    inject_mode  = "weq",
    simwidth     = 9000.0,
    simlength    = 10000.0,
    fieldwidth   = 3000.0,
    fieldlength  = 10000.0,
    n_dunes_init = 0,
    collisions   = True,
    seed         = 42,
    lambda1      = 1.0,
    lambda2_mean = 1.8,
    lambda3      = 1/3,
    alpha        = 0.05,
    delta        = 4.6,
    wind_regime  = "unimodal",
    wind_std_deg = 3.0,
)

W_eq = 4.6 * 79 / (0.25*79 - 0.05*79)
print(f"W_eq = {W_eq:.1f} m  (paper ~23m)")
print(f"n_inject/paso esperado = {37e-6 * (45*79/(W_eq+16.6)) * 0.125 * 3000:.2f}")
print()

# ── Correr 100 años primero para ver trayectoria ──────────────────────────────
N_YEARS = 100
n_steps = int(N_YEARS / params_1a["dt"])

print(f"Corriendo {N_YEARS} años ({n_steps} pasos)...")
model = DuneSwarm(**params_1a)

checkpoints = {1, 50, 100, 200, 400, 600, 800}
for i in range(1, n_steps + 1):
    model.step()
    if i in checkpoints:
        agents = list(model.agents)
        if agents:
            ws = [a.width for a in agents]
            print(f"  Paso {i:>4} ({i*params_1a['dt']:>5.1f} años) | "
                  f"N={len(agents):>4} | "
                  f"W_mean={np.mean(ws):>6.1f}m | "
                  f"W_max={max(ws):>6.1f}m")

df = model.datacollector.get_model_vars_dataframe()
years = df.index * params_1a["dt"]

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

axes[0].plot(years, df["N_dunes"], color="tab:orange", linewidth=1.5)
axes[0].set_ylabel("N dunas")
axes[0].set_title("Validación Fig.1a — qshift=0, Hersen, ρ₀=37km⁻²")
axes[0].grid(True, alpha=0.3)
axes[0].axhline(800, color="gray", linestyle="--", linewidth=0.8, label="~800 (paper 500a)")
axes[0].legend(fontsize=9)

axes[1].plot(years, df["mean_width"], color="tab:blue", linewidth=1.5)
axes[1].set_ylabel("Ancho medio (m)")
axes[1].set_xlabel("Tiempo (años)")
axes[1].grid(True, alpha=0.3)
axes[1].axhline(80, color="gray", linestyle="--", linewidth=0.8, label="~80m (paper 500a)")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f"\nEstado final ({N_YEARS} años):")
print(f"  N_dunes    = {df['N_dunes'].iloc[-1]:.0f}")
print(f"  mean_width = {df['mean_width'].iloc[-1]:.1f} m")
print(f"  calving    = {df['calving_count'].iloc[-1]:.0f}")
print(f"  colisiones = {df['collision_count'].iloc[-1]:.0f}")

In [ ]:
# Continuar desde donde quedó
N_YEARS_EXTRA = 400  # llegar a 500 años total
n_steps_extra = int(N_YEARS_EXTRA / params_1a["dt"])

print(f"Continuando hasta 500 años ({n_steps_extra} pasos más)...")
checkpoints = {1000, 1200, 1600, 2000, 2400, 2800, 3200, 4000}
for i in range(1, n_steps_extra + 1):
    model.step()
    step_total = 800 + i
    if step_total in checkpoints:
        agents = list(model.agents)
        if agents:
            ws = [a.width for a in agents]
            import numpy as np
            print(f"  Paso {step_total:>4} ({step_total*params_1a['dt']:>5.1f} años) | "
                  f"N={len(agents):>4} | "
                  f"W_mean={np.mean(ws):>6.1f}m | "
                  f"W_max={max(ws):>6.1f}m")

In [ ]:
import sys, time
from pathlib import Path
sys.path.insert(0, str(Path(".").resolve()))

from src.dune_swarm import DuneSwarm
import numpy as np

# Verificar que los dos archivos nuevos están activos
import src.flux_physics, src.collision_rules, inspect
print("flux_physics  — Shapely:", "_SHAPELY" in inspect.getsource(src.flux_physics.distribute_flux))
print("collision_rules — C-01:", "left1.intersects" in inspect.getsource(src.collision_rules.one_rule))
print()

params = dict(
    qsat=79.0, q0ratio=0.25, qshift_ratio=0.0,
    lambda2_std=0.0, outflux_mode="Hersen",
    dt=0.125, w0=16.6, c=45.0,
    rho0=37e-6, inject=True, inject_mode="weq",
    simwidth=9000.0, simlength=10000.0,
    fieldwidth=3000.0, fieldlength=10000.0,
    n_dunes_init=0, collisions=True, seed=42,
    lambda1=1.0, lambda2_mean=1.8, lambda3=1/3,
    alpha=0.05, delta=4.6, wind_regime="unimodal",
    wind_std_deg=3.0,
)

N_YEARS = 50
n_steps = int(N_YEARS / params["dt"])

print(f"Corriendo {N_YEARS} años ({n_steps} pasos)...")
t0 = time.time()
model = DuneSwarm(**params)

for i in range(1, n_steps + 1):
    model.step()
    if i % 100 == 0:
        agents = list(model.agents)
        ws = [a.width for a in agents] if agents else [0]
        elapsed = time.time() - t0
        rate = i / elapsed
        eta = (n_steps - i) / rate
        print(f"  Paso {i:>4} ({i*params['dt']:>5.1f}a) | "
              f"N={len(agents):>4} | W_mean={np.mean(ws):>6.1f}m | "
              f"W_max={max(ws):>6.1f}m | "
              f"{elapsed:.0f}s transcurridos | ETA {eta:.0f}s")

print(f"\nTiempo total: {time.time()-t0:.1f}s")
df = model.datacollector.get_model_vars_dataframe()
print(f"N_final      = {df['N_dunes'].iloc[-1]:.0f}  (paper ~300 a 50 años)")
print(f"W_mean_final = {df['mean_width'].iloc[-1]:.1f}m  (paper ~35m a 50 años)")
print(f"Colisiones   = {df['collision_count'].iloc[-1]:.0f}")
print(f"Calveos      = {df['calving_count'].iloc[-1]:.0f}")
print(f"Merging      = {df['merging_count'].iloc[-1]:.0f}")
print(f"Exchange     = {df['exchange_count'].iloc[-1]:.0f}")
print(f"Fragmentation= {df['fragmentation_count'].iloc[-1]:.0f}")

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path(".").resolve()))

import src.collision_rules as cr
import inspect

original_one_rule = cr.one_rule
call_stats = {"n_out": [], "n_colliders": [], "n_others": [], "n_ts": []}

def patched_one_rule(*args, **kwargs):
    xs, ys, lws, rws = original_one_rule(*args, **kwargs)
    call_stats["n_out"].append(len(xs))
    return xs, ys, lws, rws

cr.one_rule = patched_one_rule

# Patch en dune_swarm también
import src.dune_swarm
src.dune_swarm.one_rule = patched_one_rule

from src.dune_swarm import DuneSwarm
import numpy as np

params = dict(
    qsat=79.0, q0ratio=0.25, qshift_ratio=0.0,
    lambda2_std=0.0, outflux_mode="Hersen",
    dt=0.125, w0=16.6, c=45.0,
    rho0=37e-6, inject=True, inject_mode="weq",
    simwidth=9000.0, simlength=10000.0,
    fieldwidth=3000.0, fieldlength=10000.0,
    n_dunes_init=0, collisions=True, seed=42,
    lambda1=1.0, lambda2_mean=1.8, lambda3=1/3,
    alpha=0.05, delta=4.6, wind_regime="unimodal",
    wind_std_deg=3.0,
)

model = DuneSwarm(**params)
for _ in range(400):
    model.step()

cr.one_rule = original_one_rule

n_out = call_stats["n_out"]
print(f"Total colisiones resueltas: {len(n_out)}")
print(f"  n_out=1 (merging)      : {n_out.count(1):>5} ({100*n_out.count(1)/len(n_out):.1f}%)")
print(f"  n_out=2 (exchange)     : {n_out.count(2):>5} ({100*n_out.count(2)/len(n_out):.1f}%)")
print(f"  n_out=3 (fragmentation): {n_out.count(3):>5} ({100*n_out.count(3)/len(n_out):.1f}%)")
print(f"  n_out=0 (eliminados)   : {n_out.count(0):>5} ({100*n_out.count(0)/len(n_out):.1f}%)")
print(f"  n_out>=4               : {sum(1 for n in n_out if n>=4):>5}")

Total colisiones resueltas: 15
  n_out=1 (merging)      :    13 (86.7%)
  n_out=2 (exchange)     :     1 (6.7%)
  n_out=3 (fragmentation):     1 (6.7%)
  n_out=0 (eliminados)   :     0 (0.0%)
  n_out>=4               :     0


In [ ]:
import src.collision_rules as cr
import numpy as np

original_one_rule = cr.one_rule
sizes = []

def patched_one_rule(*args, **kwargs):
    xs, ys, lws, rws = original_one_rule(*args, **kwargs)
    for lw, rw in zip(lws, rws):
        sizes.append((lw, rw))
    return xs, ys, lws, rws

cr.one_rule = patched_one_rule
import src.dune_swarm
src.dune_swarm.one_rule = patched_one_rule

from src.dune_swarm import DuneSwarm
params = dict(
    qsat=79.0, q0ratio=0.25, qshift_ratio=0.0,
    lambda2_std=0.0, outflux_mode="Hersen",
    dt=0.125, w0=16.6, c=45.0,
    rho0=37e-6, inject=True, inject_mode="weq",
    simwidth=9000.0, simlength=10000.0,
    fieldwidth=3000.0, fieldlength=10000.0,
    n_dunes_init=0, collisions=True, seed=42,
    lambda1=1.0, lambda2_mean=1.8, lambda3=1/3,
    alpha=0.05, delta=4.6, wind_regime="unimodal",
    wind_std_deg=3.0,
)
model = DuneSwarm(**params)
for _ in range(400):
    model.step()

cr.one_rule = original_one_rule

sizes = np.array(sizes)
w_min = model.w_min
print(f"w_min = {w_min:.3f} m")
print(f"Total productos generados: {len(sizes)}")
if len(sizes) > 0:
    print(f"  lw_min = {sizes[:,0].min():.4f}  lw_max = {sizes[:,0].max():.4f}")
    print(f"  rw_min = {sizes[:,1].min():.4f}  rw_max = {sizes[:,1].max():.4f}")
    n_below = np.sum((sizes[:,0] < w_min) | (sizes[:,1] < w_min))
    print(f"  Productos con lw o rw < w_min: {n_below} ({100*n_below/len(sizes):.1f}%)")
    print(f"\nDistribución de lw:")
    for pct in [0, 10, 25, 50, 75, 90, 100]:
        print(f"  p{pct:>3} = {np.percentile(sizes[:,0], pct):.4f} m")

In [ ]:
import src.collision_rules as cr
import numpy as np

original_one_rule = cr.one_rule
empty_reasons = []

def patched_one_rule(x1, y1, lw1, rw1, x2, y2, lw2, rw2,
                     lambda1, lambda2, lambda3, alpha, delta, qshift_ratio):
    xs, ys, lws, rws = original_one_rule(
        x1, y1, lw1, rw1, x2, y2, lw2, rw2,
        lambda1, lambda2, lambda3, alpha, delta, qshift_ratio)
    
    if len(xs) == 0:
        # Diagnóstico manual
        from src.collision_rules import (
            _barchan_polys, _vs_total, _equiv_width_body,
            _equiv_width_flank, gamma_c as gc_f
        )
        from shapely.ops import unary_union
        
        l1v = _vs_total(lw1, rw1, lambda1, lambda2, lambda3)
        r1v = _vs_total(rw1, lw1, lambda1, lambda2, lambda3)
        l2v = _vs_total(lw2, rw2, lambda1, lambda2, lambda3)
        r2v = _vs_total(rw2, lw2, lambda1, lambda2, lambda3)
        vtot = l1v + r1v + l2v + r2v
        
        left1, right1 = _barchan_polys(x1, y1, lw1, rw1, lambda1, lambda2, alpha, delta)
        left2, right2 = _barchan_polys(x2, y2, lw2, rw2, lambda1, lambda2, alpha, delta)
        
        try:
            l1col = left1.intersects(left2) or left1.intersects(right2)
            r1col = right1.intersects(left2) or right1.intersects(right2)
            l2col = left2.intersects(left1) or left2.intersects(right1)
            r2col = right2.intersects(left1) or right2.intersects(right1)
            n_col = sum([l1col, r1col, l2col, r2col])
            
            # Volumen de colisionantes
            cvoltot = sum([
                l1v if l1col else 0,
                r1v if r1col else 0,
                l2v if l2col else 0,
                r2v if r2col else 0,
            ])
            cwtot = _equiv_width_body(cvoltot, lambda2, lambda3)
            
            empty_reasons.append({
                "lw1": lw1, "rw1": rw1, "lw2": lw2, "rw2": rw2,
                "n_col": n_col, "cvoltot": cvoltot, "cwtot": cwtot,
                "vtot": vtot,
            })
        except Exception as e:
            empty_reasons.append({"error": str(e)})
    
    return xs, ys, lws, rws

cr.one_rule = patched_one_rule
import src.dune_swarm
src.dune_swarm.one_rule = patched_one_rule

from src.dune_swarm import DuneSwarm
params = dict(
    qsat=79.0, q0ratio=0.25, qshift_ratio=0.0,
    lambda2_std=0.0, outflux_mode="Hersen",
    dt=0.125, w0=16.6, c=45.0,
    rho0=37e-6, inject=True, inject_mode="weq",
    simwidth=9000.0, simlength=10000.0,
    fieldwidth=3000.0, fieldlength=10000.0,
    n_dunes_init=0, collisions=True, seed=42,
    lambda1=1.0, lambda2_mean=1.8, lambda3=1/3,
    alpha=0.05, delta=4.6, wind_regime="unimodal",
    wind_std_deg=3.0,
)
model = DuneSwarm(**params)
for _ in range(400):
    model.step()

cr.one_rule = original_one_rule

print(f"Colisiones con output vacío: {len(empty_reasons)}")
if empty_reasons:
    r = empty_reasons[0]
    print(f"\nPrimer caso vacío:")
    for k, v in r.items():
        print(f"  {k} = {v}")
    
    # Estadísticas
    n_cols = [r.get("n_col", -1) for r in empty_reasons if "n_col" in r]
    import numpy as np
    print(f"\nn_col distribution: {np.unique(n_cols, return_counts=True)}")

In [2]:
import src.collision_rules as cr

original = cr.one_rule
stats = {"shapely": 0, "fallback_ncol0": 0, "fallback_exception": 0}

def patched(x1, y1, lw1, rw1, x2, y2, lw2, rw2,
            lambda1, lambda2, lambda3, alpha, delta, qshift_ratio):
    
    from src.collision_rules import _barchan_polys
    left1, right1 = _barchan_polys(x1, y1, lw1, rw1, lambda1, lambda2, alpha, delta)
    left2, right2 = _barchan_polys(x2, y2, lw2, rw2, lambda1, lambda2, alpha, delta)
    
    try:
        l1col = left1.intersects(left2) or left1.intersects(right2)
        r1col = right1.intersects(left2) or right1.intersects(right2)
        n_col = sum([l1col, r1col,
                     left2.intersects(left1) or left2.intersects(right1),
                     right2.intersects(left1) or right2.intersects(right1)])
        if n_col > 0:
            stats["shapely"] += 1
        else:
            stats["fallback_ncol0"] += 1
    except:
        stats["fallback_exception"] += 1
    
    return original(x1, y1, lw1, rw1, x2, y2, lw2, rw2,
                    lambda1, lambda2, lambda3, alpha, delta, qshift_ratio)

cr.one_rule = patched
import src.dune_swarm
src.dune_swarm.one_rule = patched

from src.dune_swarm import DuneSwarm
params = dict(
    qsat=79.0, q0ratio=0.25, qshift_ratio=0.0,
    lambda2_std=0.0, outflux_mode="Hersen",
    dt=0.125, w0=16.6, c=45.0,
    rho0=37e-6, inject=True, inject_mode="weq",
    simwidth=9000.0, simlength=10000.0,
    fieldwidth=3000.0, fieldlength=10000.0,
    n_dunes_init=0, collisions=True, seed=42,
    lambda1=1.0, lambda2_mean=1.8, lambda3=1/3,
    alpha=0.05, delta=4.6, wind_regime="unimodal",
    wind_std_deg=3.0,
)
model = DuneSwarm(**params)
for _ in range(400):
    model.step()

cr.one_rule = original

print(f"Por camino Shapely (n_col>0) : {stats['shapely']}")
print(f"Por fallback (n_col=0)       : {stats['fallback_ncol0']}")
print(f"Por fallback (excepción)     : {stats['fallback_exception']}")

Por camino Shapely (n_col>0) : 15
Por fallback (n_col=0)       : 0
Por fallback (excepción)     : 0


In [3]:
import sys, time
from pathlib import Path
sys.path.insert(0, str(Path(".").resolve()))

from src.dune_swarm import DuneSwarm
import numpy as np

params = dict(
    qsat=79.0, q0ratio=0.25, qshift_ratio=0.0,
    lambda2_std=0.0, outflux_mode="Hersen",
    dt=0.125, w0=16.6, c=45.0,
    rho0=37e-6, inject=True, inject_mode="weq",
    simwidth=9000.0, simlength=10000.0,
    fieldwidth=3000.0, fieldlength=10000.0,
    n_dunes_init=0, collisions=True, seed=42,
    lambda1=1.0, lambda2_mean=1.8, lambda3=1/3,
    alpha=0.05, delta=4.6, wind_regime="unimodal",
    wind_std_deg=3.0,
)

N_YEARS = 100
n_steps = int(N_YEARS / params["dt"])

t0 = time.time()
model = DuneSwarm(**params)

checkpoints = {100, 200, 400, 600, 800}
for i in range(1, n_steps + 1):
    model.step()
    if i in checkpoints:
        agents = list(model.agents)
        ws = [a.width for a in agents] if agents else [0]
        print(f"  Paso {i:>4} ({i*params['dt']:>5.1f}a) | "
              f"N={len(agents):>4} | "
              f"W_mean={np.mean(ws):>6.1f}m | "
              f"W_max={max(ws):>6.1f}m | "
              f"{time.time()-t0:.0f}s")

df = model.datacollector.get_model_vars_dataframe()
print(f"\nTiempo total: {time.time()-t0:.1f}s")
print(f"Colisiones   = {df['collision_count'].iloc[-1]:.0f}")
print(f"  Merging    = {df['merging_count'].iloc[-1]:.0f}")
print(f"  Exchange   = {df['exchange_count'].iloc[-1]:.0f}")
print(f"  Fragment.  = {df['fragmentation_count'].iloc[-1]:.0f}")
print(f"Calveos      = {df['calving_count'].iloc[-1]:.0f}")

  Paso  100 ( 12.5a) | N=  67 | W_mean=  20.6m | W_max=  48.3m | 3s
  Paso  200 ( 25.0a) | N=  64 | W_mean=  22.1m | W_max=  57.6m | 9s
  Paso  400 ( 50.0a) | N=  70 | W_mean=  22.7m | W_max=  72.1m | 18s
  Paso  600 ( 75.0a) | N=  78 | W_mean=  23.6m | W_max=  80.5m | 30s
  Paso  800 (100.0a) | N=  74 | W_mean=  23.5m | W_max=  99.8m | 43s

Tiempo total: 43.2s
Colisiones   = 29
  Merging    = 22
  Exchange   = 3
  Fragment.  = 4
Calveos      = 377


In [4]:
# Test rápido: mismo modelo pero wind_std_deg=0
params_fijo = {**params, "wind_std_deg": 0.0}
model_fijo = DuneSwarm(**params_fijo)
for _ in range(400):
    model_fijo.step()
df_fijo = model_fijo.datacollector.get_model_vars_dataframe()
print(f"Con viento fijo (std=0°):")
print(f"  Colisiones = {df_fijo['collision_count'].iloc[-1]:.0f}")
print(f"  Calveos    = {df_fijo['calving_count'].iloc[-1]:.0f}")
print(f"  N_final    = {df_fijo['N_dunes'].iloc[-1]:.0f}")
print(f"  W_mean     = {df_fijo['mean_width'].iloc[-1]:.1f}m")

Con viento fijo (std=0°):
  Colisiones = 10
  Calveos    = 138
  N_final    = 102
  W_mean     = 20.2m


In [5]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path(".").resolve()))

from src.dune_swarm import DuneSwarm
from src.dune_agent import DuneAgent
import numpy as np

params_fijo = dict(
    qsat=79.0, q0ratio=0.25, qshift_ratio=0.0,
    lambda2_std=0.0, outflux_mode="Hersen",
    dt=0.125, w0=16.6, c=45.0,
    rho0=37e-6, inject=True, inject_mode="weq",
    simwidth=9000.0, simlength=10000.0,
    fieldwidth=3000.0, fieldlength=10000.0,
    n_dunes_init=0, collisions=False,  # sin colisiones para aislar
    seed=42, lambda1=1.0, lambda2_mean=1.8, lambda3=1/3,
    alpha=0.05, delta=4.6, wind_regime="unimodal",
    wind_std_deg=0.0,
)

model = DuneSwarm(**params_fijo)
for _ in range(400):
    model.step()

agents = list(model.agents)
asyms  = [max(a.lw,a.rw)/min(a.lw,a.rw) if min(a.lw,a.rw)>0 else 1
          for a in agents]

print(f"Sin colisiones, viento fijo, 50 años:")
print(f"  N = {len(agents)}")
print(f"  Asimetría media  = {np.mean(asyms):.4f}")
print(f"  Asimetría máx    = {np.max(asyms):.4f}")
print(f"  Dunas con asim>2.6: {sum(1 for a in asyms if a > 2.6)}")
print(f"  Calveos totales  = {model.calving_count}")

Sin colisiones, viento fijo, 50 años:
  N = 107
  Asimetría media  = 1.0329
  Asimetría máx    = 1.8669
  Dunas con asim>2.6: 0
  Calveos totales  = 112


In [6]:
# Sin colisiones, sin inyección, una sola duna — debe tener 0 calveos
model_1 = DuneSwarm(
    qsat=79.0, q0ratio=0.25, qshift_ratio=0.0,
    lambda2_std=0.0, outflux_mode="Hersen",
    dt=0.125, w0=16.6, c=45.0,
    rho0=0.0, inject=False,  # sin inyección
    simwidth=9000.0, simlength=10000.0,
    fieldwidth=3000.0, fieldlength=10000.0,
    n_dunes_init=0, collisions=False,
    seed=42, lambda1=1.0, lambda2_mean=1.8, lambda3=1/3,
    alpha=0.05, delta=4.6, wind_regime="unimodal",
    wind_std_deg=0.0,
)

# Insertar manualmente dos dunas alineadas
from src.dune_agent import DuneAgent
a1 = DuneAgent(model_1, 11.5, 11.5)
a2 = DuneAgent(model_1, 11.5, 11.5)
model_1.space.place_agent(a1, (4500.0, 8000.0))
model_1.space.place_agent(a2, (4500.0, 2000.0))
model_1.agents.add(a1)
model_1.agents.add(a2)

for _ in range(400):
    model_1.step()

print(f"Dos dunas alineadas, sin colisiones, viento fijo, 50 años:")
print(f"  Calveos = {model_1.calving_count}")
agents = list(model_1.agents)
for a in agents:
    print(f"  id={a.unique_id} pos={a.pos} lw={a.lw:.3f} rw={a.rw:.3f} "
          f"asim={max(a.lw,a.rw)/min(a.lw,a.rw):.4f}")

Dos dunas alineadas, sin colisiones, viento fijo, 50 años:
  Calveos = 0
  id=1 pos=(4500.0, 3511.3636363636033) lw=11.500 rw=11.500 asim=1.0000
